# Naive matrix multiplication kernel

Matrix multiplication is the operation GPUs exist for. Start with the direct mapping: one thread computes one output entry,

$$C_{ij} = \sum_{k=0}^{K-1} A_{ik} B_{kj}$$

Write `matmul(A, B)`.

## Inputs and output
- `A`: `float32` `(M, K)`, `B`: `float32` `(K, N)`, with $M, N, K \ge 1$. Neither dimension needs to be a multiple of the block size.
- Return `C` as `float32` `(M, N)`.

## Requirements
- Block `(16, 16)`, grid `(ceil(N/16), ceil(M/16))`; `col, row = cuda.grid(2)` with x → column.
- Accumulate the dot product in a **local variable** (a register) and write `C[row, col]` once at the end, rather than accumulating into global memory.
- Results are compared with relative tolerance $10^{-4}$: different summation orders round differently in `float32`.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import numpy as np
from numba import cuda

BLOCK = (16, 16)

def matmul(A, B):
    """A: float32 (M, K), B: float32 (K, N). Returns A @ B as float32 (M, N)."""
    M, K = A.shape
    N = B.shape[1]
    d_A, d_B = cuda.to_device(A), cuda.to_device(B)
    d_C = cuda.device_array((M, N), dtype=np.float32)
    grid = ((N + 15) // 16, (M + 15) // 16)
    matmul_kernel[grid, BLOCK](d_A, d_B, d_C)
    return d_C.copy_to_host()

@cuda.jit
def matmul_kernel(A, B, C):
    col, row = cuda.grid(2)
    if row < C.shape[0] and col < C.shape[1]:
        # TODO: dot product of row `row` of A with column `col` of B
        C[row, col] = 0.0


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0], [3.0, 4.0]], "dtype": "float32"}, {"__nd__": [[5.0, 6.0], [7.0, 8.0]], "dtype": "float32"}], "e": {"__nd__": [[19.0, 22.0], [43.0, 50.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, 0.0, -1.0]], "dtype": "float32"}, {"__nd__": [[2.0], [3.0], [4.0]], "dtype": "float32"}], "e": {"__nd__": [[-2.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[3.0]], "dtype": "float32"}, {"__nd__": [[-2.0]], "dtype": "float32"}], "e": {"__nd__": [[-6.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.006826779805123806], [1.0461432933807373], [0.7415884137153625], [0.7239565253257751]], "dtype": "float32"}, {"__nd__": [[1.6187762022018433, -1.205558180809021, -0.6269554495811462, -1.3206632137298584, -0.10775250941514969]], "dtype": "float32"}], "e": {"__nd__": [[-0.011051028966903687, 0.008230079896748066, 0.0042800866067409515, 0.009015876799821854, 0.0007356026326306164], [1.693471908569336, -1.2611865997314453, -0.6558852195739746, -1.3816030025482178, -0.11272456496953964], [1.2004656791687012, -0.8940280079841614, -0.46494290232658386, -0.9793885350227356, -0.07990801334381104], [1.1719236373901367, -0.8727717399597168, -0.453888475894928, -0.956102728843689, -0.07800813019275665]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]], "dtype": "float32"}, {"__nd__": [[0.9987636804580688, -0.021947886794805527, 0.4958800673484802, -1.9107686281204224, 0.14706416428089142, -0.9069432616233826, 1.7753894329071045, 0.8868491053581238, 0.9493494629859924], [-0.05785496160387993, 0.6128622889518738, 0.6578901410102844, -0.3444026708602905, -0.497372031211853, -0.11477278172969818, -0.6054520010948181, -0.5943394303321838, -0.2833753824234009], [-0.7284177541732788, 0.7663277983665466, -1.5960863828659058, 0.8235621452331543, -0.6255664825439453, -0.5459399819374084, -1.3508471250534058, -0.14424212276935577, -0.2476615160703659], [0.19145582616329193, -0.5337743163108826, 0.09375617653131485, 1.8196918964385986, 0.4089996814727783, -0.5736900568008423, 0.953109622001648, -0.1288011372089386, 0.593874454498291], [0.6127474308013916, -0.39106571674346924, -1.930287480354309, -0.3476536273956299, 0.5514554381370544, -0.38011041283607483, 0.43896931409835815, 0.9795403480529785, -0.5441133975982666], [1.2313517332077026, 1.6218044757843018, 1.0790458917617798, 1.165462851524353, 1.0967963933944702, 2.2545390129089355, 0.18586090207099915, 0.0020840829238295555, 0.6031073331832886], [-0.9081951379776001, -1.553176760673523, -0.8820079565048218, 0.372565358877182, 0.47305333614349365, -1.5363587141036987, -1.883463740348816, -0.3161422312259674, -0.18805651366710663], [-0.04920083284378052, 0.6713624596595764, 1.2288341522216797, 0.23089642822742462, 0.6126847267150879, -1.0952017307281494, -1.176308035850525, 0.23623833060264587, 0.6646630764007568], [0.7262434363365173, 0.5927419662475586, 0.7841737270355225, 0.8098280429840088, -1.752185344696045, -0.7341657876968384, 0.4551597535610199, 0.5966719388961792, -1.5120923519134521], [1.1730613708496094, -0.4388319253921509, -0.2324247509241104, 0.27381864190101624, 0.7496352195739746, -1.4331170320510864, 0.9899125695228577, 0.07729347050189972, -0.770813524723053], [-0.2752731442451477, -2.185838460922241, -0.6476185321807861, 0.18190965056419373, -0.7090504765510559, -0.3280128538608551, -0.14676237106323242, 0.04838089272379875, 1.0206366777420044], [-0.02489577978849411, -0.2599679231643677, 0.5386452078819275, 2.0766170024871826, -0.34274768829345703, 0.512867271900177, -1.5773049592971802, 1.3454939126968384, 0.6593835353851318], [1.3327996730804443, -0.05906025692820549, -1.0428155660629272, 0.8860989212989807, 0.7535334825515747, -0.867038905620575, -1.4729350805282593, 0.44913211464881897, -0.17139199376106262], [2.612165927886963, -0.6142464280128479, -0.5285622477531433, -0.1107642650604248, -1.6650573015213013, -0.8890870213508606, 1.6443626880645752, -0.1313774287700653, -0.6083886027336121], [-1.2718980312347412, 0.5804831385612488, 0.6582976579666138, 0.25645434856414795, -0.389201283454895, 0.30553799867630005, 0.3815891742706299, -0.6808846592903137, 0.6223325133323669], [-2.115356206893921, -1.2890256643295288, -0.30391719937324524, -0.10849908739328384, -1.1642754077911377, 0.3287219703197479, 1.0556057691574097, 0.45104944705963135, 0.2245292216539383], [0.8282381892204285, 0.40444713830947876, -1.1423838138580322, -0.20256344974040985, 1.291327714920044, -1.137485384941101, 0.062296874821186066, -0.3891238570213318, -0.8980107307434082]], "dtype": "float32"}], "e": {"__nd__": [[0.9987636804580688, -0.021947886794805527, 0.4958800673484802, -1.9107686281204224, 0.14706416428089142, -0.9069432616233826, 1.7753894329071045, 0.8868491053581238, 0.9493494629859924], [-0.05785496160387993, 0.6128622889518738, 0.6578901410102844, -0.3444026708602905, -0.497372031211853, -0.11477278172969818, -0.6054520010948181, -0.5943394303321838, -0.2833753824234009], [-0.7284177541732788, 0.7663277983665466, -1.5960863828659058, 0.8235621452331543, -0.6255664825439453, -0.5459399819374084, -1.3508471250534058, -0.14424212276935577, -0.2476615160703659], [0.19145582616329193, -0.5337743163108826, 0.09375617653131485, 1.8196918964385986, 0.4089996814727783, -0.5736900568008423, 0.953109622001648, -0.1288011372089386, 0.593874454498291], [0.6127474308013916, -0.39106571674346924, -1.930287480354309, -0.3476536273956299, 0.5514554381370544, -0.38011041283607483, 0.43896931409835815, 0.9795403480529785, -0.5441133975982666], [1.2313517332077026, 1.6218044757843018, 1.0790458917617798, 1.165462851524353, 1.0967963933944702, 2.2545390129089355, 0.18586090207099915, 0.0020840829238295555, 0.6031073331832886], [-0.9081951379776001, -1.553176760673523, -0.8820079565048218, 0.372565358877182, 0.47305333614349365, -1.5363587141036987, -1.883463740348816, -0.3161422312259674, -0.18805651366710663], [-0.04920083284378052, 0.6713624596595764, 1.2288341522216797, 0.23089642822742462, 0.6126847267150879, -1.0952017307281494, -1.176308035850525, 0.23623833060264587, 0.6646630764007568], [0.7262434363365173, 0.5927419662475586, 0.7841737270355225, 0.8098280429840088, -1.752185344696045, -0.7341657876968384, 0.4551597535610199, 0.5966719388961792, -1.5120923519134521], [1.1730613708496094, -0.4388319253921509, -0.2324247509241104, 0.27381864190101624, 0.7496352195739746, -1.4331170320510864, 0.9899125695228577, 0.07729347050189972, -0.770813524723053], [-0.2752731442451477, -2.185838460922241, -0.6476185321807861, 0.18190965056419373, -0.7090504765510559, -0.3280128538608551, -0.14676237106323242, 0.04838089272379875, 1.0206366777420044], [-0.02489577978849411, -0.2599679231643677, 0.5386452078819275, 2.0766170024871826, -0.34274768829345703, 0.512867271900177, -1.5773049592971802, 1.3454939126968384, 0.6593835353851318], [1.3327996730804443, -0.05906025692820549, -1.0428155660629272, 0.8860989212989807, 0.7535334825515747, -0.867038905620575, -1.4729350805282593, 0.44913211464881897, -0.17139199376106262], [2.612165927886963, -0.6142464280128479, -0.5285622477531433, -0.1107642650604248, -1.6650573015213013, -0.8890870213508606, 1.6443626880645752, -0.1313774287700653, -0.6083886027336121], [-1.2718980312347412, 0.5804831385612488, 0.6582976579666138, 0.25645434856414795, -0.389201283454895, 0.30553799867630005, 0.3815891742706299, -0.6808846592903137, 0.6223325133323669], [-2.115356206893921, -1.2890256643295288, -0.30391719937324524, -0.10849908739328384, -1.1642754077911377, 0.3287219703197479, 1.0556057691574097, 0.45104944705963135, 0.2245292216539383], [0.8282381892204285, 0.40444713830947876, -1.1423838138580322, -0.20256344974040985, 1.291327714920044, -1.137485384941101, 0.062296874821186066, -0.3891238570213318, -0.8980107307434082]], "dtype": "float32"}}, {"a": [{"__nd__": [[-1.0115281343460083, 0.1010211706161499, 0.3153497576713562, 0.3090188205242157, -1.4460442066192627, -0.2887409031391144, 0.3457193970680237, 0.9468076825141907, 0.6725081205368042, 0.8316199779510498, 1.0971367359161377, 0.8239218592643738, -0.8152034282684326, 0.0580407939851284, -0.7131385803222656, -2.381474494934082], [-1.6808892488479614, 2.088918447494507, -1.1172953844070435, -2.2421035766601562, -0.5848498940467834, 1.5202916860580444, -0.044957712292671204, 0.4080832898616791, -0.27031707763671875, -0.08322902768850327, 1.2666478157043457, 0.795991837978363, -0.08625155687332153, 0.8916199803352356, 1.0744668245315552, -0.21916602551937103], [0.2699834108352661, 0.2541244626045227, -0.668296217918396, 1.1508793830871582, -0.16705165803432465, -0.6194693446159363, -1.6757957935333252, -1.2654019594192505, 1.5254234075546265, -1.1400634050369263, 0.4295547306537628, -0.3671460747718811, 0.5113277435302734, 0.6022971868515015, -0.7762848138809204, -0.29186493158340454], [0.09935615211725235, 0.11454775184392929, -0.028509825468063354, -0.31326723098754883, 0.62940913438797, 1.3171021938323975, 0.5405049920082092, -0.7401395440101624, 0.3959486782550812, 0.06110415235161781, 0.46162864565849304, 1.8448430299758911, -0.21395331621170044, -0.01150196697562933, -0.8866276144981384, -0.2889486253261566], [-0.5691503882408142, 1.5998315811157227, 0.3039744794368744, -0.5719179511070251, 1.929469108581543, -1.3281404972076416, 0.8612616062164307, 1.0162441730499268, 0.5739461779594421, -0.2611521780490875, -0.12939496338367462, -0.3434543311595917, 0.7860742807388306, -2.3289225101470947, -0.5112422704696655, 0.15194420516490936], [-0.5811653733253479, -1.8175910711288452, -0.24282878637313843, -0.7551668286323547, 0.8002673983573914, 1.3974037170410156, 0.06938022375106812, -1.934126615524292, -0.19619140028953552, -0.9241275191307068, 0.6820155382156372, 1.6914576292037964, 0.31742697954177856, 2.0352494716644287, 1.5145556926727295, 0.6965295672416687], [-0.8602482080459595, 0.2088879495859146, -1.339189052581787, 1.1509461402893066, 0.23873268067836761, 1.4988666772842407, 0.4202328026294708, -1.804278016090393, 0.7876753807067871, 1.07448410987854, 1.80921471118927, -0.8258050084114075, -0.7756575345993042, 0.21190856397151947, -0.9085264801979065, -1.231732964515686], [-0.9525958895683289, 2.729684352874756, -0.4994106888771057, -0.8910261392593384, -0.33179616928100586, 0.11256977915763855, 1.39730966091156, 1.0500785112380981, 0.8921658992767334, 0.14716559648513794, 0.2590750753879547, -0.5361987352371216, -1.4042506217956543, 2.1393494606018066, -0.35637354850769043, 1.1124544143676758], [-0.21796570718288422, 0.006845590192824602, -0.05694355070590973, -1.2813628911972046, 0.016955828294157982, 0.4752615988254547, 0.9226380586624146, 0.5869653224945068, 1.1122077703475952, 0.38568171858787537, 0.08443869650363922, -1.8780494928359985, -0.4487834870815277, 1.6402403116226196, 0.2076982855796814, 0.8823252320289612], [-0.1269516795873642, -0.588998019695282, 2.132202386856079, 0.4422747492790222, 1.5792279243469238, 0.614720344543457, -0.211176335811615, -2.160973310470581, 0.17426158487796783, -1.4840788841247559, -2.1390092372894287, 1.3231945037841797, 0.8046885132789612, 0.7238641381263733, 0.1307019144296646, 0.044714074581861496], [0.5039088726043701, 0.23595571517944336, 0.15106749534606934, 0.8205365538597107, -2.380427598953247, 0.970963180065155, -0.5569902062416077, 0.9304091334342957, 1.248805284500122, -0.7729722261428833, 0.8611832857131958, 0.845461368560791, -0.6570132970809937, -0.4895789623260498, 0.14685191214084625, -2.2022597789764404], [-0.42555034160614014, 0.204661026597023, -0.43219590187072754, -1.5870498418807983, -0.3796837329864502, 1.5282191038131714, 1.336464524269104, -0.11038530617952347, -0.7978920340538025, -0.008755696937441826, 0.02163836546242237, -0.6720709204673767, -0.33005383610725403, 0.6292916536331177, 0.9403054118156433, -2.4363632202148438], [0.9827693700790405, 0.140268474817276, 1.1554882526397705, -0.015401363372802734, 0.4446943402290344, 1.645554780960083, -2.967830181121826, 1.5954985618591309, 0.19392716884613037, -1.820354700088501, -2.665189266204834, 0.3153252601623535, -1.025542974472046, 1.0073479413986206, 0.2184889167547226, -0.13121718168258667], [0.9977638125419617, 0.19632619619369507, 0.7452183961868286, 0.8350223302841187, 0.32414525747299194, 0.7923579216003418, 0.7910439968109131, -1.3673619031906128, -0.2716832756996155, -0.11908839643001556, -0.37679043412208557, -1.5874146223068237, -0.5663399696350098, 0.3403848111629486, 0.19391073286533356, 1.525384545326233], [-0.6055780053138733, -0.43163007497787476, -0.5719561576843262, -0.3152042627334595, -0.6029431223869324, -1.8704246282577515, 0.34609222412109375, 0.9899774789810181, 1.1215486526489258, -1.086402416229248, -0.1924332231283188, 1.8507945537567139, -0.10718041658401489, 0.3493174910545349, -0.15161240100860596, 1.00685715675354], [0.9690619111061096, 2.3283443450927734, 0.5000448822975159, 0.44955340027809143, -1.2126399278640747, -0.40945035219192505, -0.7017499208450317, -0.32820987701416016, -1.3108655214309692, 1.2170063257217407, -1.0536633729934692, 0.27364587783813477, -1.2599124908447266, 0.3472849726676941, 1.658156156539917, -0.8199522495269775]], "dtype": "float32"}, {"__nd__": [[0.3651025891304016, -0.06676550209522247, 0.2838982045650482, 0.42082464694976807, -0.6355125308036804, -0.17983348667621613, -0.5890219211578369, -0.8246515393257141, -1.2282038927078247, 0.9654253125190735, -0.7381125688552856, 0.6658728122711182, 0.5552874803543091, -0.9985285997390747, 1.0484603643417358, -0.860573947429657], [0.6288726925849915, 1.4581091403961182, -0.7153669595718384, -0.9289090037345886, -0.9425963759422302, 2.296931266784668, -0.3755772113800049, -1.2651017904281616, 0.8140742778778076, -0.17906174063682556, -0.036196451634168625, -1.001160740852356, -2.230409860610962, -0.7368532419204712, -1.0534553527832031, -1.6287273168563843], [-1.096008539199829, -0.9584379196166992, 0.7948848605155945, 0.6957101821899414, 0.5054454207420349, -0.5447176694869995, -1.2700574398040771, 0.2862849235534668, 0.44877889752388, 0.99150550365448, -0.03853512555360794, -0.14054352045059204, 0.29785746335983276, -0.9381491541862488, -0.8154915571212769, 0.6310967803001404], [-1.2702391147613525, -0.08994454890489578, 0.7725995779037476, 0.2277238816022873, 0.4222502112388611, -1.1163036823272705, -0.13038785755634308, -0.27552834153175354, 0.5996634364128113, 0.4176621735095978, 0.030023127794265747, 1.9038910865783691, 0.549006462097168, -0.279000461101532, -0.60347580909729, -0.02495349943637848], [-0.5580549836158752, 1.6507090330123901, 0.5846701860427856, -0.2925161123275757, -0.6363402605056763, -1.0530509948730469, -0.25471729040145874, 1.3329887390136719, 0.8634799718856812, -0.031118402257561684, 0.1654348373413086, -1.8381857872009277, 0.7873908281326294, -0.19260276854038239, 1.4041026830673218, -0.14910189807415009], [-0.453822523355484, 0.24651966989040375, -0.6041839122772217, 1.3769683837890625, 0.3224976658821106, 0.405113160610199, 0.3735569715499878, -0.31189990043640137, 0.5237529277801514, 0.2183353453874588, 1.8016003370285034, 0.10837627202272415, 0.4236889183521271, -0.4044346213340759, -0.31671440601348877, -0.24615290760993958], [1.3584250211715698, 0.15237902104854584, 1.047925353050232, -0.18186751008033752, -0.2815110683441162, -1.8408570289611816, 0.9155233502388, 2.1493167877197266, -0.1378524899482727, 1.5268951654434204, 0.9436737895011902, -0.5898204445838928, 0.8596398830413818, -0.8002117276191711, -0.20229357481002808, 1.3184913396835327], [0.25872159004211426, -1.0722317695617676, 1.5129715204238892, -0.9987009167671204, -0.06137297675013542, -0.24582748115062714, -0.06141023710370064, -0.7510008215904236, -1.534639835357666, 1.9540082216262817, -0.8465635180473328, -0.785310685634613, -2.3394389152526855, 0.18721865117549896, 1.3868184089660645, 1.4974733591079712], [2.862344980239868, 1.5983045101165771, 0.5099833607673645, 0.12953783571720123, 1.5378870964050293, 0.07011284679174423, 0.30401086807250977, -0.851283609867096, 0.2062998116016388, 2.0784735679626465, 0.0342593751847744, 0.5032802224159241, -2.2979631423950195, 1.4491254091262817, 0.4762026071548462, 1.5106031894683838], [0.3398790955543518, -0.7565504312515259, -1.2698602676391602, 0.5875890254974365, 0.05402841791510582, -0.8987361192703247, 0.0031042355112731457, -0.5717014074325562, 0.7732594013214111, -0.3608226478099823, -1.1796208620071411, -0.05065413936972618, -0.5296242833137512, 1.4976699352264404, -0.447028249502182, -1.3058639764785767], [-0.053283728659152985, -0.2588012218475342, 0.2587713301181793, 0.19976456463336945, -1.0507824420928955, -0.07786799967288971, -0.13400542736053467, -1.2310553789138794, -0.6087012887001038, 0.15898244082927704, -1.6367652416229248, -1.1338529586791992, -0.5229032635688782, -0.1538432240486145, 0.44379064440727234, -1.4276379346847534], [0.22262096405029297, 0.5446657538414001, 1.4627785682678223, -0.5013926029205322, -0.9983723759651184, -1.6165105104446411, 0.46724218130111694, 1.2032893896102905, -0.7701380252838135, 0.36400216817855835, 0.7867556810379028, 0.887794554233551, -0.45764705538749695, 0.5948933362960815, -0.0861329510807991, -0.9718559980392456], [0.24396894872188568, 0.3505101799964905, -0.9644555449485779, 0.4460042119026184, 2.0001769065856934, -1.4851642847061157, 0.7156851887702942, 0.9294318556785583, 0.0692492127418518, 0.3447718024253845, -0.24582205712795258, 1.3695982694625854, -0.9262272715568542, -0.12906435132026672, -1.9349712133407593, 0.3206176459789276], [0.36625197529792786, -1.6833674907684326, -0.8852362036705017, 0.4567379951477051, -0.5142275094985962, -1.2390056848526, -2.532902479171753, 0.3074023127555847, 1.8940067291259766, -0.5038554072380066, 0.386272132396698, 0.3684244155883789, -0.13572202622890472, 0.04430411010980606, 0.06186873838305473, -0.9240309596061707], [2.4435524940490723, 2.6667943000793457, 0.2581786513328552, 1.7343430519104004, -0.8728395700454712, -0.27874359488487244, 1.0808945894241333, -1.2030354738235474, -1.352702021598816, 0.2929803729057312, -2.027817726135254, 1.6777607202529907, 0.7075417637825012, 0.207941472530365, 0.6394781470298767, -0.9348348379135132], [-0.9643130302429199, 0.8150201439857483, 0.922928512096405, 0.13009577989578247, -0.058569539338350296, -0.5469236373901367, 1.386376142501831, -1.4913914203643799, -0.8635812401771545, -0.5437981486320496, -1.5328112840652466, -1.3319623470306396, -1.0333994626998901, 1.00994873046875, 1.8032090663909912, 0.34855592250823975]], "dtype": "float32"}], "e": {"__nd__": [[3.3174986839294434, -7.15155029296875, 0.3826005458831787, -2.714695692062378, -0.28560250997543335, 0.9563241004943848, -3.7219996452331543, 1.1686515808105469, 1.3050771951675415, 4.1548566818237305, 2.7191762924194336, 2.112942695617676, -3.9351484775543213, 0.945167064666748, -5.111373424530029, 0.14204609394073486], [6.903543472290039, 4.287832260131836, -4.453103542327881, -0.09681854397058487, -5.664072513580322, 6.940300941467285, 1.224738597869873, -3.7111692428588867, 0.5047091245651245, -3.0478827953338623, 0.5598154067993164, -4.9327850341796875, -7.381720542907715, 1.2380951642990112, -1.510384202003479, -7.1540656089782715], [-0.09644785523414612, 1.409239411354065, -2.8375706672668457, -0.6444570422172546, 3.800205945968628, 3.4973573684692383, -3.3733627796173096, -3.3213272094726562, 3.902466297149658, -1.8075568675994873, 0.7316566705703735, 4.453526973724365, -2.619492769241333, 0.9589151740074158, -2.1049675941467285, -1.2557464838027954], [-0.2728290557861328, 1.271904468536377, 1.363724708557129, -0.30695652961730957, -1.7461994886398315, -2.6399266719818115, 0.2852569818496704, 4.571836948394775, 1.9046554565429688, 0.9385759830474854, 6.454187393188477, -1.482079029083252, 0.7120807766914368, -0.051786914467811584, -0.9409019351005554, -2.35945463180542], [1.5780220031738281, 7.947751522064209, 4.264547348022461, -6.7219014167785645, 1.3595439195632935, 2.4624757766723633, 5.31148624420166, 2.573983669281006, -2.449462413787842, 4.530984401702881, -1.8075679540634155, -8.806650161743164, -6.674801349639893, -0.9708425998687744, 0.8242552876472473, 5.448283672332764], [1.6998822689056396, 3.861346483230591, -0.2209545373916626, 7.220246315002441, -2.8498587608337402, -8.790786743164062, 0.24352848529815674, 5.100654602050781, 1.8074870109558105, -4.3862223625183105, 2.70133113861084, 3.661792278289795, 8.642082214355469, 1.5217632055282593, 2.156820058822632, -4.601078033447266], [0.3093440532684326, -0.22306561470031738, -6.137800216674805, 2.0176842212677, -0.11970201134681702, 2.204866409301758, -1.193368911743164, -0.37794697284698486, 8.186328887939453, -3.013171434402466, 4.528814792633057, -0.8202617168426514, 3.06716251373291, 0.9717158079147339, -3.9704442024230957, -5.044233322143555], [6.319802284240723, -0.012080967426300049, -0.5715093612670898, -4.137521266937256, -4.82649564743042, 4.9751877784729, -3.652553081512451, -4.513627529144287, 4.631221294403076, 1.3203752040863037, 0.37448060512542725, -9.563045501708984, -11.133831024169922, 1.233629822731018, 2.8905537128448486, -0.8818840980529785], [5.83477258682251, -1.3535912036895752, -2.3234317302703857, 1.8191730976104736, 0.952440083026886, 0.7027287483215332, -2.2304399013519287, -3.0853123664855957, 2.6702046394348145, 1.9578685760498047, -1.705177664756775, -5.628317356109619, -3.7385876178741455, 1.9236369132995605, 4.33478307723999, 3.4833619594573975], [-4.181060314178467, 4.233661651611328, 1.4966055154800415, 3.640702486038208, 3.5207154750823975, -6.258461952209473, -2.8039963245391846, 10.092785835266113, 6.312218189239502, -1.2122581005096436, 9.091924667358398, 5.3242645263671875, 8.600162506103516, -3.2046658992767334, -4.114655494689941, 1.9534894227981567], [5.093400478363037, -2.6717348098754883, 1.724332332611084, 0.39747801423072815, 0.875946044921875, 5.244174957275391, -1.7500183582305908, -4.586094379425049, -3.5499281883239746, 6.770302772521973, 2.9410176277160645, 8.260337829589844, -4.112227916717529, -1.470036268234253, -3.9721336364746094, 0.21326589584350586], [6.127701759338379, -1.3375988006591797, -5.366586685180664, 2.7381293773651123, -2.6871588230133057, 2.957038164138794, -1.9234379529953003, 4.616117477416992, 2.335454225540161, -0.09535026550292969, 5.891514778137207, 0.2931177616119385, 5.600592136383057, -4.279779434204102, -4.215085506439209, -1.9101660251617432], [-4.48581600189209, -1.0212572813034058, 1.8696321249008179, 0.7953275442123413, 0.7393369078636169, 6.435988426208496, -7.0841875076293945, -4.324986934661865, 0.2894599437713623, 1.0196013450622559, 5.275016784667969, 3.1802656650543213, -1.7967548370361328, -1.999731183052063, 4.511876106262207, 2.319141387939453], [-3.371427536010742, 1.624971866607666, -0.8528110980987549, 4.217190265655518, -0.1892998218536377, 0.007475435733795166, -0.41248106956481934, -2.2527408599853516, 2.7834129333496094, -1.5842187404632568, -0.32369279861450195, -1.4192956686019897, 5.916494846343994, -3.1343207359313965, 1.7604079246520996, -0.2968495488166809], [4.46998405456543, 0.9855283498764038, 7.37004280090332, -5.254079341888428, -0.308304101228714, -3.926236152648926, 2.4959139823913574, 1.5674293041229248, -5.047699928283691, 3.428457736968994, -1.4893441200256348, 0.3004411458969116, -6.84684944152832, 4.471724510192871, 4.247672080993652, 5.084615230560303], [1.9641377925872803, 1.0862246751785278, -3.846545934677124, 1.5050063133239746, -6.724822998046875, 6.764317512512207, -3.856853485107422, -6.265615940093994, 0.6749783158302307, -3.4715845584869385, -3.3303470611572266, 4.948954105377197, 0.7108485698699951, -2.4359312057495117, -3.620321273803711, -10.348861694335938]], "dtype": "float32"}}, {"a": [{"__nd__": [[-1.6207902431488037, -0.6070793271064758, -0.1365053802728653, -0.2689482271671295, -0.3710603713989258, 0.5167067050933838, -1.9433256387710571, -0.5751184821128845, 0.21701933443546295, 0.7194798588752747, 0.29128819704055786, 0.02727508544921875, -0.12797845900058746, -0.6535109877586365, 0.10379975289106369, -0.1860288381576538, -0.6831533908843994, 1.227534532546997, 1.2133013010025024, -0.343110054731369], [0.6478995680809021, 1.2303869724273682, 0.3010129928588867, 0.16319222748279572, 0.533564031124115, -1.6094402074813843, -1.388628363609314, 0.5641837120056152, 0.31798651814460754, 1.4107551574707031, -0.45128875970840454, 1.1433027982711792, 1.1052647829055786, 2.0170841217041016, -0.7537370324134827, -0.30582287907600403, 0.3243243992328644, 0.31543371081352234, -0.946397066116333, -0.5178493857383728], [-1.2503142356872559, 2.35825777053833, -0.014660049229860306, 0.04179604724049568, -0.5613086819648743, 0.07734435796737671, 0.8396328687667847, -0.27927303314208984, 0.022781088948249817, 2.2216477394104004, 0.9740800261497498, 0.28827720880508423, -0.36944103240966797, 0.02870151400566101, -0.6910870671272278, 0.32132962346076965, -0.622902512550354, -0.6811097264289856, -0.3657000958919525, -1.0320606231689453], [0.22902335226535797, 2.544227123260498, -0.5966071486473083, -1.6360853910446167, 0.9733805656433105, -0.09233435988426208, 1.9833978414535522, 0.38283586502075195, 0.9271625876426697, -0.7799530029296875, -1.601825475692749, -0.19799187779426575, 0.6464803814888, -0.8833605647087097, -1.6160253286361694, -0.6256598234176636, -0.4793476164340973, -0.3675890564918518, -0.07663442194461823, 2.340507984161377], [-0.020807888358831406, -0.5484915971755981, -1.1372740268707275, 0.7408779263496399, -1.637664556503296, 1.382768154144287, -0.0692056268453598, -1.5625590085983276, 0.11900428682565689, -0.18023833632469177, 1.1559903621673584, -1.5623806715011597, 0.13309185206890106, -0.08598951995372772, 0.8455734252929688, -1.2040666341781616, -0.8482613563537598, 1.8892676830291748, 0.48163822293281555, -1.5463167428970337], [0.24978017807006836, -0.8602015376091003, -0.7618016600608826, -1.2919890880584717, 0.9046835899353027, -0.6118576526641846, 0.12099337577819824, -0.001280653872527182, -1.5914479494094849, -0.8183493614196777, 0.6869032979011536, -1.6409785747528076, 0.1693340539932251, -0.1818251758813858, 0.5669850707054138, 0.6073947548866272, 0.7467851042747498, -0.015694335103034973, 0.18280354142189026, -0.7752683162689209], [-0.6121394634246826, 0.8266044855117798, -0.6652244329452515, 0.9528847932815552, 0.9287781715393066, -0.8623022437095642, 0.49295368790626526, -0.8760589361190796, 0.2750808596611023, -0.8388142585754395, 0.2715766727924347, 0.10641057789325714, -1.0994683504104614, 1.9858564138412476, 0.9012876749038696, -1.653531789779663, 0.2916918396949768, -0.4793999493122101, 1.9918155670166016, -0.1088230237364769], [0.3436782956123352, -0.22924132645130157, 0.5835829377174377, -0.5805071592330933, -0.9420105814933777, -0.5401186943054199, -0.4660601019859314, 0.5821076035499573, 0.045021288096904755, -0.484965056180954, -0.4468144178390503, -0.9529879689216614, 0.08830008655786514, -2.0678298473358154, -0.26045891642570496, 1.143375277519226, -0.39920881390571594, -0.690424382686615, 0.007050355896353722, -0.9650694131851196], [-0.7926846742630005, 0.646466851234436, -0.6684660315513611, 1.9174853563308716, 0.5028208494186401, -0.13717034459114075, -0.6520593166351318, -0.2902238667011261, -2.2982258796691895, 0.20977440476417542, 0.8532809019088745, 0.5375636219978333, -0.4639601409435272, 0.5814618468284607, 0.6041796803474426, -0.16340264678001404, 0.31649190187454224, 1.003752589225769, 0.42431679368019104, -0.41006818413734436], [-0.019297847524285316, 0.12086328864097595, -1.6313539743423462, 0.09883256256580353, -0.15777890384197235, -2.1931893825531006, -1.7379909753799438, 1.7670197486877441, 0.07323595881462097, 1.3134818077087402, -0.40190109610557556, 1.5042898654937744, 0.5411241054534912, -0.7092797756195068, 0.25254809856414795, 1.5410654544830322, 2.3647758960723877, 0.49363210797309875, -0.4657880961894989, -0.8821107149124146], [0.6686070561408997, 1.0769553184509277, 0.12849944829940796, -0.11871745437383652, 0.8069870471954346, 0.4799248278141022, 0.38778698444366455, -1.531358242034912, -0.33262479305267334, 1.8839209079742432, -0.017489293590188026, 2.4250705242156982, 0.3860815465450287, -0.2843066155910492, 0.28332558274269104, 0.6050259470939636, -0.7185158729553223, 0.24642819166183472, -0.41389867663383484, 1.456837773323059], [-0.45136627554893494, 0.12149367481470108, -1.4990419149398804, -1.3628976345062256, -0.08908195793628693, -0.912798285484314, -0.29296770691871643, 0.23536138236522675, 0.5608831644058228, -1.4064985513687134, -0.18523308634757996, 0.6628106236457825, -0.6755565404891968, -1.3018429279327393, -0.6911688446998596, 1.6249867677688599, 0.019243337213993073, 1.5576485395431519, -0.3059655427932739, 1.1903440952301025], [-1.4431207180023193, -0.2517355978488922, 0.0887589156627655, 1.7581918239593506, 0.47164666652679443, 1.1205030679702759, -0.4179016649723053, -0.4134337902069092, 1.276160478591919, 1.4298580884933472, 1.2315402030944824, 1.2054142951965332, -0.5174822211265564, 2.161595344543457, 0.1769961267709732, 1.277896523475647, 0.7573860883712769, -1.7472846508026123, -1.9365215301513672, -1.5948790311813354], [-0.04182733967900276, 1.1247016191482544, -0.24081912636756897, 0.8498364686965942, -1.2288641929626465, -0.6161066889762878, -0.8932043313980103, 0.03756294026970863, -1.1453760862350464, -0.09723257273435593, 0.5986259579658508, -0.4566332995891571, -0.8892152309417725, -0.4487883448600769, -1.2170778512954712, -1.0313078165054321, 0.8077372312545776, 0.34457990527153015, -1.4373092651367188, -0.27767789363861084], [0.45804598927497864, 0.8067469000816345, 0.6462011337280273, 0.9866752028465271, -0.03348594903945923, -0.010324367322027683, -0.24027857184410095, 1.1055976152420044, 0.86436527967453, 1.7919553518295288, 1.1291881799697876, -0.8049775958061218, -1.4704294204711914, -1.3869999647140503, -0.8901602625846863, 0.6221355199813843, -1.398931622505188, -2.0275955200195312, -0.18017280101776123, 0.8103318214416504], [0.7783640623092651, 1.6696728467941284, -0.04075911268591881, -0.5676541328430176, -0.3149374723434448, -1.3509571552276611, -0.6871153712272644, 1.2713083028793335, 2.0870535373687744, 1.2558772563934326, 0.277999609708786, -0.2637236416339874, 0.22365696728229523, -1.9159985780715942, 0.5535761117935181, 0.7602910399436951, -0.16047969460487366, -0.007142330054193735, -1.728898286819458, 0.7600390911102295], [0.7612931728363037, -2.8039090633392334, -0.6144441962242126, 0.3160645067691803, 1.6434342861175537, -0.897066593170166, -0.06976022571325302, 1.4434279203414917, 0.4459795355796814, -0.24850910902023315, -0.4735415577888489, 0.00547418650239706, -0.8240422606468201, 0.9944770932197571, -0.8909828066825867, 0.5521201491355896, 0.9262480735778809, 1.2096810340881348, 0.13121895492076874, 0.04314431920647621], [0.37378185987472534, 0.13793087005615234, 1.1592787504196167, -0.03946287930011749, 2.0018231868743896, 2.878631114959717, -0.5757231712341309, 0.4800769090652466, -1.0872809886932373, 0.8262948989868164, 0.4883621335029602, -0.14645451307296753, -0.980353832244873, 0.09311962127685547, 2.082486867904663, 1.6763986349105835, 0.4491090774536133, -0.5581597089767456, 0.03344980254769325, -0.7479337453842163], [-1.3040378093719482, -0.9047393798828125, 0.6596301198005676, 0.3534374237060547, 0.222939595580101, -1.0661869049072266, -1.3766030073165894, 1.054458498954773, -0.32438671588897705, 1.2291778326034546, -0.7126662731170654, 1.0171236991882324, 0.2773536741733551, -0.40095263719558716, -0.3969977796077728, 1.9622247219085693, 0.9690816402435303, -0.22939735651016235, 0.20397426187992096, -1.5250656604766846], [1.7636947631835938, 1.4387168884277344, -0.3733363747596741, 0.49503713846206665, -2.743748664855957, 1.0129203796386719, 0.5072152614593506, 0.29578685760498047, 0.8617136478424072, 0.32965999841690063, -0.9540830850601196, -0.6791681051254272, -0.1671738475561142, 0.9187893271446228, 0.033025506883859634, -0.6687888503074646, 0.36458849906921387, -1.2371844053268433, 2.0902340412139893, 0.3041323721408844], [-0.9918118119239807, 0.1653609275817871, 1.0811634063720703, 0.5418669581413269, 2.046400785446167, -1.430374026298523, 1.59821617603302, -0.030150150880217552, 0.8851730823516846, 1.7239842414855957, -1.2072172164916992, -1.4238739013671875, -0.39219579100608826, -1.67239511013031, -2.028059482574463, 0.0828106701374054, 1.1702712774276733, -0.4803355634212494, 0.5063671469688416, 3.7003912925720215], [0.33989688754081726, -0.30407771468162537, -0.15893203020095825, 0.4383302628993988, -0.28045031428337097, 0.7026527523994446, 0.22796742618083954, 0.9329825043678284, 0.16055557131767273, -0.07021734118461609, -0.9240994453430176, -1.3797935247421265, -0.2753864526748657, -0.37907177209854126, 0.29467928409576416, -0.06446235626935959, -0.38039880990982056, 2.2856788635253906, 0.333213746547699, 1.2338004112243652], [-1.0182632207870483, 1.0267860889434814, 1.1926556825637817, 0.46514901518821716, -0.8794736266136169, 1.2759158611297607, -0.7142456769943237, -1.5524861812591553, -1.6184920072555542, 0.0761510580778122, -0.4841650426387787, 0.8186705708503723, 0.6846855282783508, 1.388148546218872, 0.3500062823295593, 0.12404786795377731, 0.0185148473829031, -0.782452404499054, -0.8257942795753479, -0.7246132493019104], [0.24196356534957886, -0.8459450602531433, -0.08269441872835159, 1.0909847021102905, -2.4004313945770264, -0.8813552856445312, -1.0011082887649536, -0.017222408205270767, -1.5087724924087524, -0.6855587363243103, 2.2890212535858154, -1.392246127128601, -0.10802508145570755, 0.18838471174240112, -0.3216644823551178, -1.2334669828414917, 0.9765813946723938, -1.9539461135864258, -1.004881501197815, -0.2650070786476135], [-0.6743685007095337, 1.1327903270721436, -0.3687936067581177, 1.8582794666290283, -0.07349009811878204, 1.4815893173217773, 0.3523347079753876, -0.3277737498283386, -0.3709101676940918, 0.9451366662979126, -0.6341208815574646, -0.12591299414634705, -0.8905877470970154, -2.1460671424865723, 0.45054635405540466, 0.5463460683822632, -1.309014916419983, 0.3803046643733978, -0.3406950831413269, -2.010200262069702], [-1.0410434007644653, 0.5744715332984924, 0.9488037824630737, 0.3046136796474457, 2.719693422317505, 0.2697148621082306, -0.8357396721839905, -1.561798095703125, 1.4264661073684692, -0.5485516786575317, -0.44824305176734924, -0.894351065158844, -0.8875692486763, -0.42912110686302185, 0.40670207142829895, 0.49541473388671875, 0.30270326137542725, 0.9096903204917908, -0.530950129032135, -1.0648069381713867], [-0.08619792014360428, -2.100137948989868, 0.8329634070396423, 0.49311164021492004, 0.5154929757118225, -1.109553337097168, -0.5145267844200134, 0.006326213479042053, -1.0342342853546143, -1.1130638122558594, -0.22759301960468292, 0.13391607999801636, -0.6849114894866943, 1.0204204320907593, 2.2150113582611084, 0.56794273853302, -0.19743338227272034, -0.3364275097846985, 1.6077959537506104, 0.2963317632675171], [0.6029301881790161, 0.21655571460723877, -0.580817461013794, 1.5149461030960083, -0.25122976303100586, 0.14492911100387573, -2.1662094593048096, 2.1688625812530518, 0.749665379524231, -0.10631189495325089, -0.8069295287132263, -0.6279671788215637, -1.7276605367660522, 0.45854899287223816, -0.8904040455818176, 0.41639000177383423, 0.44272348284721375, 1.918687343597412, 0.20702548325061798, 0.29000747203826904], [0.4997800588607788, -0.6273570656776428, -0.24772188067436218, 0.3288741707801819, 1.1695114374160767, 0.7586452960968018, -0.2560211420059204, 1.2427797317504883, -0.3582799434661865, -1.5069586038589478, 1.6011186838150024, 0.053651247173547745, -0.42276161909103394, -1.1690495014190674, -0.39344578981399536, 1.7438205480575562, -0.003692975966259837, 2.291738986968994, 0.09204798936843872, 1.2054623365402222], [0.10599732398986816, -0.3860933184623718, -0.35456424951553345, -0.19222098588943481, -1.1376323699951172, -1.1683454513549805, 0.4411744475364685, 1.3318233489990234, 0.24051730334758759, 0.13340361416339874, -0.8314495086669922, 1.223784327507019, 1.1429508924484253, -1.0779972076416016, 0.16020716726779938, -0.24521951377391815, -1.411595106124878, 1.7082703113555908, 1.2814476490020752, 0.41311076283454895], [1.5143764019012451, -0.4176057279109955, 0.37335845828056335, 1.451432704925537, -0.20017263293266296, 0.40036723017692566, -0.8777526021003723, 0.74681556224823, 1.2572336196899414, -0.8922179341316223, -0.7063461542129517, 0.012422759085893631, 1.4856841564178467, -0.43539613485336304, -0.7635531425476074, -0.03694044426083565, 1.2121126651763916, -1.4701186418533325, -0.10776050388813019, 1.3748573064804077], [0.026222793385386467, -0.22882935404777527, 0.910961389541626, -1.16195547580719, 2.109994649887085, 0.5278990268707275, -0.340266615152359, 0.12397429347038269, -0.028819162398576736, -0.25823071599006653, -1.198325514793396, -0.2213895469903946, -1.7024714946746826, 1.3982822895050049, -1.1151864528656006, -0.7248526215553284, -0.7888097763061523, 0.7288885712623596, -0.19481757283210754, 1.4411065578460693], [-0.4564025104045868, 0.15170465409755707, -0.06433016806840897, -0.3393384516239166, 1.1191164255142212, 1.2320133447647095, 1.3152743577957153, -1.3846955299377441, 0.34880343079566956, -1.1250537633895874, -1.1540749073028564, -1.8066492080688477, -0.9469943642616272, 0.5971924066543579, 1.3933337926864624, 1.7366218566894531, -0.9088893532752991, -0.62497878074646, 0.23381179571151733, 0.14638040959835052]], "dtype": "float32"}, {"__nd__": [[0.21683269739151, -1.7558437585830688, -0.1917571872472763, 0.5811179280281067, 0.2627290189266205, -0.12169016897678375, -0.7283618450164795, -0.09596467763185501, 1.598081350326538, 0.5000199675559998, 1.050817608833313, 1.6577023267745972, -1.092559576034546, 0.6654503345489502, 0.4558957517147064, 1.3194738626480103, -0.2506585717201233, -0.22914209961891174], [-0.6712613105773926, 0.16569986939430237, -0.29970407485961914, -0.1750430166721344, 0.31033098697662354, -0.40463578701019287, -0.9547361135482788, -0.43729105591773987, -0.5777490735054016, 0.39655739068984985, 1.2153106927871704, -0.37362444400787354, -2.4811384677886963, 0.5508542656898499, -0.0963561162352562, -0.977351725101471, -0.28662821650505066, 1.8213351964950562], [-1.5302897691726685, -1.2015324831008911, -0.1012866348028183, -3.1564059257507324, -0.5021199584007263, 0.9325849413871765, 0.5000510215759277, 0.3314838111400604, -0.379700243473053, -0.2041529417037964, 2.700169563293457, 0.1921415627002716, 1.3062770366668701, -1.297217845916748, 0.20505096018314362, 0.3216400146484375, 0.1129157617688179, -2.2110097408294678], [-0.4623335599899292, 1.4162425994873047, 0.5128253102302551, -0.8221522569656372, 0.8350256085395813, -1.9172033071517944, 0.5429336428642273, 0.3051203787326813, -0.6554907560348511, -1.061988353729248, -1.7054922580718994, -0.26474544405937195, 1.6901241540908813, 0.8207033276557922, 0.36737215518951416, 0.881934642791748, -0.6353544592857361, 0.4841739535331726], [-2.960207223892212, -0.5367698073387146, -0.5290698409080505, -0.11811430752277374, -0.22369860112667084, 0.5351437926292419, -0.6771001219749451, 0.8683037161827087, -0.30621349811553955, 0.8143633008003235, 0.05671912804245949, 0.7890329957008362, 0.39972609281539917, -0.666083574295044, 0.12150222063064575, -1.3694981336593628, -0.017791127786040306, 0.311030775308609], [-0.34467971324920654, -0.2341393679380417, 0.9117961525917053, 1.1353224515914917, 1.3491140604019165, -0.6400141716003418, -0.4952453672885895, 0.7218685746192932, 0.821160614490509, -1.8898893594741821, 1.9070384502410889, 1.0879770517349243, -1.3105263710021973, 1.0912139415740967, -0.4777120351791382, -0.18046726286411285, -1.0385234355926514, -0.8726392388343811], [-0.4378573000431061, 0.03843201696872711, -1.1587274074554443, -0.11689814180135727, -0.7569604516029358, -0.5753968954086304, -0.9321466088294983, -0.6727166771888733, 1.5534213781356812, -1.6627708673477173, -1.3616470098495483, -0.11698245257139206, -0.5469444990158081, -1.099422574043274, 0.3564349114894867, -1.2202856540679932, -0.05590137466788292, 0.8983364701271057], [0.5127048492431641, -0.34055498242378235, -0.04877311363816261, 0.11923643201589584, 0.5483090877532959, -1.0067095756530762, 1.213614583015442, 0.698826014995575, 1.8094396591186523, 0.7877311110496521, 0.97388756275177, -3.0021302700042725, -0.4951585829257965, 1.808349847793579, 0.3313179910182953, -0.5075802803039551, 0.6419144868850708, 0.03552979603409767], [0.34793251752853394, -0.7729684114456177, 0.4752761721611023, 1.4662350416183472, 0.18924443423748016, 0.39185938239097595, -0.7509913444519043, 0.4325178861618042, -0.46052637696266174, -1.4111883640289307, -0.4720814526081085, -0.28956305980682373, 0.5053873062133789, 0.8983536958694458, -0.25583145022392273, -0.4177341163158417, 0.5826492309570312, -0.9610499739646912], [-0.3944273293018341, -0.29621997475624084, -0.22451692819595337, -0.655421793460846, 1.57233726978302, 0.900712788105011, -0.226342111825943, -1.4948683977127075, -0.5629835724830627, 0.5350067019462585, -2.0096664428710938, 2.431035280227661, -0.1337328553199768, -0.10784193128347397, 0.5805734992027283, -1.4851104021072388, 0.010916383005678654, -0.06093188747763634], [0.6593376994132996, -1.0658069849014282, 0.5744242072105408, 1.1407345533370972, 0.24506236612796783, -0.5953289270401001, -0.90166836977005, -1.8310890197753906, -0.6247636079788208, 0.4538174867630005, 1.4533023834228516, 0.9886080026626587, 1.1533124446868896, -0.5231764912605286, -0.48014000058174133, -1.5870686769485474, -0.7214457988739014, -1.1827975511550903], [-0.29860931634902954, 0.6269626021385193, -0.16664943099021912, -0.37793323397636414, 0.6422966718673706, 1.469225525856018, 1.0370961427688599, -1.7469978332519531, 1.7020901441574097, -0.4866539537906647, -1.4902021884918213, -0.40406614542007446, 0.9589853882789612, -2.7109649181365967, 1.2685226202011108, 0.1769101470708847, 0.5204325914382935, -1.4156651496887207], [-0.5974912643432617, -1.181147813796997, 1.1375162601470947, -0.8209843039512634, 0.33391693234443665, -0.703159511089325, -1.7093653678894043, 0.8417617082595825, 1.1508452892303467, 1.4705592393875122, -0.5997622609138489, 0.0057991985231637955, 0.6533916592597961, 0.2907525897026062, 0.17052514851093292, 2.3016016483306885, -0.46364283561706543, 0.8447886109352112], [-0.5776507258415222, 1.3826419115066528, -0.30266475677490234, -1.0296095609664917, 0.7139660716056824, 0.011213802732527256, -0.4232196807861328, -0.26393839716911316, -0.6602922081947327, 0.48032835125923157, 0.29142120480537415, 1.6321558952331543, -0.7219659686088562, 1.39801025390625, 0.5168749690055847, -1.455296277999878, 1.4689761400222778, -1.8169453144073486], [0.4751683473587036, -1.904505968093872, 0.24149012565612793, -0.40489017963409424, -0.303010493516922, -1.9501585960388184, -0.7945665717124939, 1.1664409637451172, 0.6116440892219543, -0.07760929316282272, -0.13617683947086334, -0.654072105884552, 1.4576730728149414, -0.02580845169723034, 1.1066291332244873, -1.475903034210205, -1.538569450378418, 1.3947741985321045], [-0.8556200861930847, -0.16525880992412567, 0.5144577026367188, 0.9657150506973267, 0.679697573184967, 0.6018398404121399, 1.6811884641647339, 0.3071710169315338, 0.9640346765518188, 0.09466089308261871, 0.2147883027791977, 1.9644849300384521, -0.20489592850208282, -2.4991812705993652, 0.12888412177562714, 0.24227218329906464, 1.8951948881149292, -0.3300327658653259], [-1.1175780296325684, 0.16975536942481995, 1.9072641134262085, -0.30745965242385864, 1.3187636137008667, 0.5421028137207031, 0.6842325329780579, -0.8178902268409729, 0.09459694474935532, -0.7989511489868164, 0.01924891769886017, -0.6980023384094238, 1.1192229986190796, -0.5391094088554382, -0.24402707815170288, -2.2660350799560547, -1.016312599182129, 2.440908432006836], [-0.42804017663002014, -1.6606796979904175, -0.05491599440574646, -0.2768874168395996, -0.5556930303573608, -0.3477404713630676, -0.524111807346344, 0.8935447931289673, -1.2616474628448486, 0.16763857007026672, 0.7991078495979309, -0.951500415802002, 2.2355806827545166, 0.0074569471180438995, -0.15179884433746338, -0.6127159595489502, 0.743439793586731, -1.6614123582839966], [0.5362709760665894, 2.184621810913086, 0.30408817529678345, 0.8100376129150391, -0.9549872875213623, 0.768435001373291, -0.7973754405975342, -0.4597005546092987, -0.0075430115684866905, 1.9233665466308594, -0.5961700677871704, -1.9628545045852661, 0.13280364871025085, 0.7919993996620178, 0.6784860491752625, 0.24577677249908447, 1.4062124490737915, 0.9212743043899536], [0.8789829015731812, -0.3942983150482178, -1.4385101795196533, 1.6713608503341675, -0.8352416753768921, 1.3425437211990356, 1.23598313331604, -1.1300537586212158, -0.5996738076210022, 0.4785591959953308, 1.0359755754470825, -0.909664511680603, -0.23810994625091553, -0.9842091202735901, -1.173655390739441, 0.14655497670173645, -0.4319477677345276, -2.001530647277832]], "dtype": "float32"}], "e": {"__nd__": [[3.090866804122925, 1.7655447721481323, 2.8940463066101074, 1.609805941581726, -0.6956160068511963, 2.041618585586548, -0.3077252507209778, 1.3014841079711914, -7.421737194061279, 3.3527259826660156, -0.7449551820755005, -2.4897964000701904, 6.342032432556152, 1.5294209718704224, -0.7878623604774475, 1.1373209953308105, 1.9318691492080688, -4.147738933563232], [-5.815926551818848, -0.047906532883644104, 0.14238744974136353, -8.782655715942383, 5.92755651473999, 3.4564530849456787, -0.44511932134628296, -2.50137996673584, -2.096031665802002, 6.700962066650391, -2.9850902557373047, 5.190027236938477, -0.4453429579734802, 2.070481061935425, 3.424990177154541, -1.0383962392807007, 4.46666145324707, -2.876746654510498], [-1.5538396835327148, 3.9128530025482178, -1.2200943231582642, -2.3024919033050537, 4.901411533355713, 0.4396789073944092, -2.5491392612457275, -7.484086513519287, -2.8621792793273926, -1.7518064975738525, -4.362161636352539, 7.5221028327941895, -7.324025630950928, -1.7554028034210205, 0.9457252025604248, -7.060922145843506, 0.7414753437042236, 3.324831485748291], [-1.276653528213501, -0.4033755660057068, -8.766046524047852, 7.155972480773926, -6.081231594085693, 6.161625862121582, -2.741243362426758, -1.1197110414505005, 3.021956205368042, 0.4839160442352295, 2.7681572437286377, -7.538778781890869, -15.429027557373047, -0.5106925964355469, -4.881528377532959, 2.170583963394165, 0.7815996408462524, 3.4075827598571777], [7.143688201904297, -2.111131191253662, 4.379994869232178, 2.8688747882843018, -0.9680918455123901, -10.795370101928711, -10.499743461608887, 4.214487075805664, -6.3767619132995605, -3.6690828800201416, -0.4662933349609375, 2.166307210922241, 4.748851776123047, 10.655205726623535, -1.170973300933838, 1.3220891952514648, -4.581566333770752, 0.9612066745758057], [-1.071845293045044, -2.8949332237243652, 1.8362867832183838, 1.5856205224990845, -2.857053279876709, -2.640316963195801, -2.0344150066375732, 3.31672739982605, 1.2110960483551025, 5.47707986831665, 2.909989833831787, 1.1491634845733643, 0.2578027546405792, -0.27000582218170166, -0.9738778471946716, -2.7766706943511963, -1.0007450580596924, 8.08156681060791], [-0.44891518354415894, 11.03538990020752, -2.0629231929779053, -0.8413545489311218, -3.6376402378082275, -1.961229920387268, -5.763364791870117, -2.668466091156006, -6.219435691833496, 1.8323755264282227, -5.301641941070557, -5.391078948974609, 0.7370033860206604, 6.003784656524658, 2.8163537979125977, -8.588767051696777, 0.6371762752532959, 5.1730194091796875], [3.214733600616455, -2.9427595138549805, 2.1145315170288086, 0.7042073011398315, -2.0144808292388916, -0.47723448276519775, 3.22556734085083, 4.287949085235596, 3.7461278438568115, 1.0234060287475586, 2.5526633262634277, -2.509798288345337, -1.8920738697052002, -1.1683592796325684, -0.8869035243988037, 8.317934036254883, 1.047337293624878, 4.7784271240234375], [-2.802170515060425, 5.699443817138672, 1.2746000289916992, -4.287943363189697, 2.1783318519592285, -5.216089248657227, 0.7541376352310181, -2.0619773864746094, -4.825445175170898, 3.0354695320129395, -3.1872503757476807, -0.6004480719566345, 5.764258861541748, -0.3667813241481781, 2.332242012023926, -4.666470050811768, -1.940908670425415, 3.635871410369873], [-0.9428116679191589, -0.43412718176841736, 6.747602939605713, 0.15423941612243652, 7.912894248962402, 2.058764934539795, 8.070113182067871, -2.5731053352355957, 4.101738929748535, 5.706228256225586, -10.481738090515137, -4.558058738708496, 6.300932884216309, -4.338648319244385, 3.7919158935546875, -3.4491989612579346, 3.2938694953918457, 10.319028854370117], [-4.516782760620117, -3.5875906944274902, -4.584693908691406, 0.15236139297485352, 2.4777987003326416, 7.112964153289795, -0.40296876430511475, -7.709069728851318, 1.8241941928863525, 0.03246331214904785, -4.049767017364502, 10.447663307189941, -1.2187559604644775, -12.67949390411377, 2.404602527618408, -0.7037839889526367, -0.9861082434654236, -6.036921501159668], [3.6326146125793457, -2.2257087230682373, -2.1481804847717285, 10.936052322387695, -4.435332775115967, 5.343972206115723, 6.4446821212768555, -0.13142728805541992, 0.24799573421478271, 0.7659848928451538, 1.2669346332550049, -6.9513139724731445, 0.3657103180885315, -8.113133430480957, -3.8464202880859375, 1.2468037605285645, 5.4199676513671875, -2.5902833938598633], [-7.059561729431152, 4.99816370010376, 6.0165605545043945, -4.042626857757568, 13.36457633972168, -1.7918845415115356, 3.1459858417510986, -3.8167037963867188, -1.514440655708313, -10.136088371276855, -6.422595977783203, 16.277116775512695, 2.6337361335754395, -2.782876968383789, 2.735938310623169, -10.2489652633667, -0.12744241952896118, -3.550438404083252], [2.6724400520324707, 1.9518510103225708, 0.9339934587478638, -2.6485915184020996, 2.3215177059173584, -2.4409303665161133, 3.299976110458374, -3.5861427783966064, -5.847080230712891, -1.7029452323913574, 2.724889039993286, -1.3228490352630615, -1.4204721450805664, 3.128931760787964, -3.417715072631836, -0.3078533411026001, -4.276126384735107, 2.7391903400421143], [3.237741470336914, 0.5060343742370605, -3.78999924659729, 4.907881259918213, 2.381023406982422, 1.4654840230941772, 4.545230865478516, -5.169336318969727, -1.2596242427825928, -0.6065710783004761, 1.7697741985321045, 4.594056606292725, -7.378593921661377, 1.070380449295044, -2.13315486907959, 0.3023616075515747, -0.37202396988868713, -2.903399705886841], [2.725222110748291, -12.451160430908203, -0.9196501970291138, 4.364861965179443, 1.8668668270111084, 0.29422152042388916, 1.679105520248413, -0.2785724997520447, 0.7412675619125366, 0.6873485445976257, 1.77727472782135, -0.9889019131660461, -2.0859076976776123, -1.4351329803466797, -1.7787644863128662, -0.7503031492233276, -2.093393087387085, 1.846565842628479], [-3.4269261360168457, 0.8272017240524292, -0.25495654344558716, 1.9602303504943848, 0.2178756296634674, 3.0451102256774902, 5.8614373207092285, 2.6912589073181152, 1.2828079462051392, 1.4884681701660156, -3.3551416397094727, -1.9088096618652344, 7.810280799865723, 0.6572127938270569, 0.7522993087768555, -2.671649217605591, 7.161116600036621, -5.990599632263184], [-9.340954780578613, -5.618344783782959, 3.7565858364105225, -1.5224915742874146, 6.522601127624512, -2.958099365234375, 0.9786132574081421, 4.529182434082031, 4.204712390899658, -2.304300308227539, 9.852758407592773, 9.914112091064453, -1.0136550664901733, -2.7088122367858887, 2.8170735836029053, -9.515985488891602, -3.651982545852661, 0.7233958840370178], [-5.386261463165283, 3.991856336593628, 5.547414779663086, -6.77817440032959, 5.567234039306641, 4.363264560699463, 8.983765602111816, 0.5954498052597046, 1.9431027173995972, 3.3738889694213867, -6.953161239624023, 0.17780017852783203, 7.122828006744385, -6.8671369552612305, 3.962181806564331, 0.39462655782699585, 6.563316345214844, 2.55253005027771], [8.872576713562012, 7.351325511932373, 1.3531135320663452, 4.174228191375732, 2.3343236446380615, -2.2921714782714844, -3.546251058578491, -3.001770496368408, 3.6309165954589844, -1.1015640497207642, -0.0136660635471344, -3.237316131591797, -12.095932960510254, 12.90259838104248, 1.3565906286239624, 3.1452248096466064, 0.4143078923225403, 5.970481872558594], [-6.6661272048950195, 2.387625217437744, -7.64863395690918, 2.1225552558898926, -4.663938999176025, 12.091275215148926, 6.1037983894348145, -5.331193447113037, -6.880684852600098, 0.794752836227417, -3.966315507888794, -4.325547695159912, -0.306130588054657, -7.643396377563477, -6.38487434387207, -3.536436080932617, 0.5534660816192627, 0.7735080718994141], [2.4598207473754883, -4.51017951965332, -2.1476478576660156, 3.3071794509887695, -3.018503189086914, -3.567157506942749, 0.35019421577453613, 6.218976020812988, -2.2191667556762695, -0.2937546372413635, 4.411128997802734, -6.952556133270264, 1.969308614730835, 6.20570182800293, -2.4563286304473877, 1.0379152297973633, 0.6250976920127869, -3.6342966556549072], [-4.342965602874756, 4.3238959312438965, 2.712975025177002, -10.434605598449707, 4.719849586486816, -0.8581022024154663, 0.19757819175720215, -0.47809040546417236, -1.5466936826705933, -2.7185192108154297, 2.225853443145752, 8.613347053527832, -2.292421340942383, -2.8857386112213135, 1.7381772994995117, 1.7542396783828735, -2.6632890701293945, -0.889358401298523], [9.583357810974121, 3.1228137016296387, 4.539349555969238, -1.7572840452194214, 2.065293312072754, -6.556067943572998, 2.38974928855896, -5.471826076507568, -3.012831449508667, 0.15685886144638062, 2.843531370162964, 0.8092598915100098, 1.18772554397583, 5.946413516998291, -3.283132553100586, 2.879267692565918, -8.039999008178711, 3.752304792404175], [-1.830278992652893, 1.198905348777771, 1.3428258895874023, -1.2351067066192627, 3.131686210632324, -7.994908332824707, -0.9870096445083618, 4.710376262664795, 0.0024918317794799805, -7.5189361572265625, -5.568939208984375, 2.5417184829711914, -1.2514125108718872, 0.8832566738128662, 1.9768232107162476, 1.7241346836090088, -3.3543143272399902, 7.626736164093018], [-11.499361991882324, -3.432659864425659, 2.002838134765625, -2.8470263481140137, -1.161522626876831, 0.683294415473938, -2.666161060333252, 8.108694076538086, -9.369674682617188, -4.136287212371826, 3.447031021118164, 4.494400978088379, 5.626760482788086, -2.415632963180542, -1.62165105342865, -5.97845458984375, -0.4150691032409668, 0.9008328914642334], [0.7362090945243835, 2.7459871768951416, -0.825070858001709, -3.959075927734375, -5.457401752471924, -0.949687123298645, 3.8718230724334717, 3.448936939239502, 0.7086091041564941, 5.203469753265381, -0.8155643343925476, -4.412552833557129, 9.755001068115234, -3.0835988521575928, 4.60128116607666, 0.12968885898590088, 2.7677969932556152, -2.1657943725585938], [1.9241337776184082, 2.546380043029785, 1.625800609588623, 3.7859079837799072, 3.6175878047943115, -1.6873115301132202, 7.830621719360352, 4.477745532989502, -5.017395496368408, 0.84662926197052, 4.790811538696289, -7.578014373779297, 1.1866376399993896, 10.731151580810547, -1.7129653692245483, -0.7062572836875916, 5.592978000640869, -4.838961124420166], [-1.3272826671600342, -7.142642498016357, 0.5098533630371094, 8.93159008026123, -2.0821573734283447, -1.0129691362380981, 4.765661239624023, 2.596331834793091, 1.3704359531402588, 1.1790801286697388, 10.415807723999023, -4.7059006690979, 6.640574932098389, -5.186177730560303, -3.6117568016052246, -0.18298593163490295, 2.1738061904907227, -8.069748878479004], [6.4134521484375, -1.7656949758529663, -3.0557825565338135, 0.7351431846618652, -5.145211219787598, 0.13723981380462646, -0.03763550519943237, 1.0575134754180908, 4.896387577056885, 5.736526012420654, -6.069552898406982, -12.848313331604004, 4.667881965637207, -0.29866519570350647, 3.18129563331604, 8.456680297851562, 5.11183500289917, -2.6169869899749756], [0.36593472957611084, -0.3693539500236511, 4.383340835571289, 2.6159560680389404, 3.3830440044403076, 0.4164159297943115, 3.4141454696655273, 1.6350985765457153, 4.972248077392578, -1.2087831497192383, 1.8609126806259155, -3.505746841430664, -0.31201913952827454, 3.7642264366149902, -2.006113052368164, 10.123623847961426, -3.078711986541748, 0.8034191131591797], [-5.513539791107178, 0.9512264728546143, -8.157333374023438, -0.615347146987915, -3.961243152618408, 8.22846508026123, 2.7331526279449463, 1.960170030593872, -5.516884803771973, 1.0742263793945312, 8.623434066772461, 1.3101164102554321, -4.754173278808594, 0.9122176170349121, -2.7669119834899902, -3.826770544052124, 4.775218486785889, -12.826292037963867], [-3.69075083732605, 0.5574424862861633, -2.8609044551849365, 3.9151365756988525, -4.141659736633301, -2.7103424072265625, -2.2649505138397217, 8.079249382019043, -0.5896106958389282, -5.136342525482178, 2.915282964706421, 6.350268840789795, -6.266071796417236, -1.1201711893081665, -1.066733956336975, -2.8534324169158936, 1.2892659902572632, 2.6475603580474854]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.45285260677337646, -0.34919387102127075, -0.36104562878608704, -1.305973768234253, 0.7621538043022156, 0.34251725673675537, 0.33784887194633484, -0.032815393060445786, 0.39300623536109924, 1.6368945837020874, 0.2158578783273697, 0.2713713049888611, -0.4224463105201721, 0.6992930173873901, -0.20751667022705078, -0.6975704431533813, 1.1799633502960205, -0.13298141956329346, -0.8426026105880737, -0.6801744699478149, -0.49495747685432434, -1.0551775693893433, 1.4937522411346436, 0.6305328011512756, -1.339452862739563, 1.2190451622009277, -0.33762603998184204, 0.29815804958343506, -0.35177144408226013, 0.45618879795074463, 0.4209728538990021, 0.8008323311805725, 0.8652925491333008, -2.562810182571411, -0.49552690982818604, -0.030108265578746796, -0.22051376104354858, -1.4859005212783813, -0.4100390076637268, -0.2299884557723999, -2.1070330142974854, -0.9300584197044373, 0.6647905111312866, -0.8265393972396851, -0.2607077360153198, -0.08303412795066833, 0.42388466000556946, 0.6543334126472473, 0.9844890832901001, 0.050063081085681915, 0.14042632281780243, -0.19409169256687164, 0.1514560580253601, -0.16060034930706024, 0.7123098373413086, 0.6702373027801514, -1.4188956022262573, -2.3148086071014404, 0.39779093861579895, -0.564548909664154, -1.5932608842849731, 0.9180218577384949, 1.472454309463501, -1.1882528066635132], [-0.9947476983070374, -0.728964626789093, -0.8520814180374146, -0.1739819496870041, -1.1845386028289795, -0.07617136836051941, -0.4615330696105957, 0.9904285073280334, -0.1643318384885788, 0.8316735625267029, 1.4442594051361084, 0.4629114270210266, 0.5916503071784973, -0.5523177981376648, -0.05300980433821678, 2.231823444366455, 1.3181943893432617, -0.022946318611502647, -0.4628274142742157, -0.2182755023241043, 1.0325888395309448, -2.1303281784057617, 0.18389302492141724, -1.3941205739974976, 1.3641657829284668, -1.5089468955993652, 0.8528716564178467, -0.36054128408432007, -0.05049913376569748, 1.5982354879379272, -0.07297597080469131, 0.6222567558288574, 0.32483112812042236, -1.2279624938964844, -0.10203304141759872, 0.9243291616439819, 0.29423022270202637, 0.142941415309906, -0.7761984467506409, 1.50875723361969, -0.6150681972503662, 1.7613097429275513, 0.14366653561592102, 0.6646720767021179, -1.4728481769561768, 0.0678243488073349, -0.9997174143791199, 0.042774129658937454, 1.1779807806015015, 0.32194632291793823, 0.21946634352207184, 0.9494724273681641, -1.2347759008407593, 0.08149854093790054, -0.6368495225906372, 1.0904308557510376, -0.7293274402618408, 1.61894953250885, 2.0462493896484375, -2.1419179439544678, -0.16121628880500793, -1.6071466207504272, 1.2489351034164429, -0.8755391836166382], [0.6402547955513, 0.3233199417591095, -0.9351303577423096, -1.5319688320159912, 1.1362340450286865, -0.5994064807891846, 0.2796059548854828, 1.4481185674667358, -0.02086012437939644, -0.2970115840435028, -1.0241578817367554, 0.47864893078804016, -0.8194818496704102, -1.6968138217926025, -0.5854759812355042, -0.6285947561264038, -1.6387577056884766, -0.24163182079792023, 0.8253733515739441, 0.5830301642417908, 2.0538527965545654, 1.1533024311065674, 0.27049556374549866, -2.051737070083618, 0.23835061490535736, 1.248002290725708, -0.26118388772010803, 1.467392921447754, -0.9163438081741333, 0.2859095633029938, 0.8643002510070801, 1.745025873184204, -0.027281546965241432, 2.012868881225586, -0.33955711126327515, -1.8012216091156006, -0.0913446918129921, 1.427912712097168, -0.3252072334289551, 1.699069619178772, -0.18983186781406403, 1.9579267501831055, -2.237377405166626, 2.361107349395752, -2.076385259628296, -0.04690681770443916, -1.2026305198669434, 0.3396807610988617, -0.21001200377941132, -0.29385876655578613, 0.6640295386314392, 1.0036770105361938, -0.3269942104816437, 1.2825323343276978, -1.7723937034606934, -1.0857235193252563, -1.3975104093551636, 0.5027144551277161, 0.9465919137001038, -0.4336540997028351, -0.6543287038803101, -1.2309291362762451, 0.7034623622894287, 1.180850863456726], [-0.5725268125534058, -0.18454155325889587, -0.41525813937187195, -0.024824490770697594, 0.47712746262550354, -2.287346601486206, -0.5560782551765442, 0.840452253818512, -0.9375386834144592, 1.3124055862426758, 1.4642820358276367, -0.9698680639266968, -0.8127599358558655, 1.366702675819397, -1.6880539655685425, -1.0798512697219849, -1.6740059852600098, 0.03451698273420334, 0.37485310435295105, 0.5115864872932434, 0.29838553071022034, -1.4415862560272217, 1.2241945266723633, -0.05640263482928276, -0.40353742241859436, 1.7823007106781006, 0.13240185379981995, 0.36842912435531616, 0.8706337213516235, 2.0885188579559326, -0.011337807402014732, 0.7155918478965759, -2.414926290512085, -1.0125716924667358, 2.132798671722412, 2.0614871978759766, -0.1602429747581482, -0.9295006990432739, -1.4200125932693481, -1.249221920967102, -0.6034155488014221, -1.7964167594909668, -0.12086116522550583, -0.18422484397888184, -1.2887921333312988, -1.3583381175994873, -1.2237995862960815, -0.5172356963157654, -0.2386845350265503, 0.576640784740448, -1.823046326637268, -0.3737705945968628, 0.020918352529406548, 0.31990760564804077, 0.8712254166603088, -0.431165486574173, 1.272396206855774, 2.363239288330078, 0.45351386070251465, 1.6058502197265625, 1.0133702754974365, 0.47266602516174316, -0.20818471908569336, 0.280451238155365], [2.137233257293701, 0.09740959852933884, -0.7756903767585754, -0.4873889684677124, 0.8298765420913696, -0.6295126080513, 0.9718260765075684, 0.7871955633163452, 0.2722683548927307, 1.7469630241394043, -1.3071112632751465, -0.52965247631073, -0.14232036471366882, 0.4404275715351105, -1.3568748235702515, -0.5568231344223022, 1.3695729970932007, -0.3843144476413727, 0.9704785346984863, -0.22517114877700806, -0.7605617642402649, 0.5406985878944397, 1.3532543182373047, -0.04516865685582161, -1.3474949598312378, -0.8027077913284302, -0.4683149456977844, -0.2783759534358978, -0.9786970615386963, 0.6951465010643005, -1.1066774129867554, 0.9954793453216553, -1.1028670072555542, -1.0214848518371582, 1.1402727365493774, 0.7137534022331238, -0.37055671215057373, -0.6371582746505737, -0.2550383508205414, 0.025817666202783585, 0.23466400802135468, 1.05502188205719, -0.8717917203903198, -0.012797416187822819, 0.04590341076254845, -1.4247705936431885, 1.1780256032943726, 1.7011005878448486, -0.029088303446769714, -1.8278040885925293, -0.8650181889533997, 0.9317935109138489, -2.121798038482666, -2.088683605194092, -0.5659490823745728, 0.43614521622657776, 0.8059864640235901, 0.994572103023529, -2.479867458343506, -0.6535250544548035, -1.0508933067321777, -1.0628314018249512, 0.7410619854927063, 0.3769517242908478]], "dtype": "float32"}, {"__nd__": [[-0.8110707402229309, 0.179348424077034, 0.16444708406925201, 1.252562165260315, -0.6093438267707825, 0.5969393253326416, -0.7754384279251099, 0.4100613296031952, -0.0745493471622467, 0.23306602239608765, -0.0328940786421299, 1.5097869634628296, -0.7212711572647095, -0.9464458227157593, -0.5606027245521545, 0.4755491018295288, 0.7335492968559265, -0.029837114736437798, -0.6366634368896484, -0.3746419847011566, -0.6504840850830078, 0.5401875376701355, 0.030921846628189087, 0.2769450545310974, 0.11272543668746948, 0.1250387579202652, 1.1899447441101074, -0.4269334077835083, 0.46638885140419006, 0.5002380609512329, -0.6709368824958801, 1.6555776596069336, 0.2637922763824463, -1.0844900608062744, 0.9758313298225403, 0.5836494565010071, 0.7211081981658936], [-0.17790457606315613, -0.3269049823284149, 0.1721903681755066, -1.3149988651275635, 0.1804748922586441, 0.08600326627492905, 0.891893208026886, -0.19433997571468353, 0.6043950915336609, -1.3808529376983643, -0.7384414076805115, 0.771332859992981, 1.4351773262023926, 0.8791624903678894, -1.460036039352417, -1.0092531442642212, -0.6004521250724792, 1.212814211845398, -0.5943936109542847, 0.2292768657207489, 0.45303860306739807, 0.4493517577648163, -0.14210213720798492, 0.3217971920967102, 0.8299887776374817, 0.16342703998088837, -0.0438787117600441, 0.7741922736167908, 1.4869515895843506, 2.421678066253662, -1.1015777587890625, 0.28049466013908386, 0.11524848639965057, -1.0621085166931152, -0.2679346203804016, -0.6555896997451782, -0.9516890048980713], [-0.5908293128013611, 0.6209826469421387, -1.0058672428131104, -0.5165213346481323, -0.1342669129371643, -0.9778062701225281, -1.2104113101959229, -1.5698050260543823, -0.7512903213500977, 1.3514931201934814, -1.6809816360473633, 0.8016745448112488, -0.9266989231109619, -0.9503595232963562, 0.7260758876800537, 0.5119954943656921, 0.5951291918754578, -0.038861457258462906, 0.8100455403327942, -0.6468110084533691, 0.30686917901039124, 0.911801278591156, -0.07593902200460434, -0.8705646991729736, 1.6006686687469482, -0.24896368384361267, -0.9869053959846497, 0.5866063237190247, 1.2159343957901, 0.530538022518158, -1.193668007850647, 0.03735867887735367, 0.28367623686790466, 0.6005132794380188, -1.4132448434829712, -1.2623822689056396, 0.17128725349903107], [-0.0058206007815897465, 0.42795535922050476, 1.121259093284607, -1.3527488708496094, -1.1324657201766968, -0.5967116951942444, 1.4623523950576782, -0.2260216921567917, 0.9282556176185608, -2.358680248260498, 0.19236259162425995, 0.8817595839500427, 1.3680357933044434, -0.7117049098014832, 1.9709362983703613, -0.8067092299461365, 1.43987238407135, 1.6885236501693726, -0.06222442910075188, -1.3039573431015015, 2.1902248859405518, 0.39672353863716125, -1.1606539487838745, 0.5096933245658875, -1.0199801921844482, 0.186734139919281, -1.0681583881378174, -2.232119560241699, 0.6769204139709473, -0.6864755749702454, -0.1322779804468155, 0.10380671918392181, -0.1797959804534912, -0.7042406797409058, 0.3502923846244812, 0.5908114314079285, 0.2180754542350769], [-0.3461383283138275, 0.04066858068108559, 1.4997482299804688, 1.2082773447036743, -0.2176831215620041, 0.6395557522773743, 0.6282811760902405, 0.6355608701705933, 0.11390183866024017, 0.561543881893158, -0.24232250452041626, 0.7795019149780273, 0.6198180913925171, 0.7655924558639526, 0.7521952390670776, 0.649696946144104, -0.671213686466217, -1.0633028745651245, 0.23354457318782806, -0.2938297390937805, 0.8166251182556152, -0.5469607710838318, -0.37298667430877686, -1.2465788125991821, -0.040121015161275864, -0.3186190128326416, 1.2038516998291016, 0.1334499567747116, 1.452333688735962, -0.6729598045349121, -1.4934651851654053, 1.137908697128296, 0.5735691785812378, 1.6311947107315063, -0.5732730627059937, -1.6181780099868774, 1.65132737159729], [0.40461477637290955, -0.44756123423576355, 2.251404285430908, 0.19894307851791382, -0.6520024538040161, 0.5418161749839783, -0.35230860114097595, -0.6470746994018555, 0.5709282159805298, 0.21203455328941345, -0.013582125306129456, 0.0660555437207222, -1.1619669198989868, 0.872577965259552, 2.048837423324585, 0.45112496614456177, 1.430090069770813, 2.1245992183685303, -0.04515114426612854, -0.4990198612213135, -0.7774618864059448, -0.29994919896125793, 0.9351500868797302, 0.021569037809967995, 1.330027461051941, -0.4666799306869507, 1.788069248199463, -0.14985500276088715, 0.07851483672857285, 0.2290380746126175, -0.06615365296602249, 0.04612858220934868, 0.7370700836181641, 0.04387878626585007, 0.5903343558311462, -0.6272169947624207, -0.628275990486145], [-1.2366372346878052, 0.8508766293525696, -0.6815167665481567, 0.5681730508804321, 0.23690585792064667, 0.255741685628891, -0.09051335602998734, 0.6947554349899292, 0.6180436015129089, 1.7505989074707031, -0.07805158197879791, -0.22083891928195953, -1.039297103881836, 2.2797927856445312, -0.5448080897331238, 0.9436092376708984, 0.8953094482421875, -1.0478838682174683, -0.48460400104522705, 0.8574503064155579, 0.1287371963262558, -1.1242998838424683, 0.2534976601600647, 1.6097699403762817, 0.4627222418785095, 0.658879280090332, 1.3751745223999023, 2.2381625175476074, 0.14958342909812927, -0.4888996183872223, 0.04209807515144348, -0.36115333437919617, -1.2359371185302734, -0.0522715225815773, -0.8488275408744812, 1.316775918006897, -0.027204303070902824], [-2.4847311973571777, -0.9449190497398376, 0.42304566502571106, -0.5237311720848083, -0.7664869427680969, -0.5215431451797485, -0.8112435340881348, 0.6022043228149414, -0.4617481231689453, -0.9097607731819153, 0.8149917721748352, -1.1824990510940552, 0.1963101476430893, -1.0152857303619385, -0.45287075638771057, -0.5269129276275635, -1.1582475900650024, -0.6258463859558105, 1.409229040145874, -1.1520973443984985, 1.923327088356018, 0.7680197358131409, 1.504771113395691, -0.5341224074363708, 1.0267001390457153, 0.15772511065006256, 1.9828872680664062, -0.4695587754249573, 1.1952793598175049, -0.6281103491783142, 1.3571192026138306, -0.40312182903289795, -1.1484602689743042, -1.404089093208313, -0.6204761266708374, 1.4863368272781372, -1.3571559190750122], [-0.8125250935554504, -0.16616196930408478, -0.02393212541937828, 0.1758386790752411, 1.5428482294082642, -0.26535001397132874, -1.4988244771957397, 0.6869074106216431, -0.4683501720428467, -0.27681654691696167, 0.0843040719628334, 1.5430943965911865, 1.1900734901428223, 0.009455546736717224, -0.38542357087135315, 0.4168659448623657, -0.7917735576629639, -1.7978401184082031, -0.6958492994308472, 0.9443269968032837, -0.7083731889724731, -0.5366072058677673, 1.424930453300476, 1.4378467798233032, 1.4653691053390503, -0.7878955006599426, 0.4864814281463623, 1.3405190706253052, 1.0167568922042847, -0.38760411739349365, -0.2207241803407669, -0.7556382417678833, -0.5417242646217346, 0.3614385426044464, 0.36010587215423584, -0.8790721893310547, 0.647200345993042], [0.18881194293498993, 1.280922532081604, -0.5415319800376892, 0.2166769802570343, -2.2421460151672363, 0.7664321660995483, 0.5756988525390625, 0.8553410172462463, 0.0402025543153286, 0.034097589552402496, -0.05460840463638306, 1.2482218742370605, -0.9493243098258972, -0.07737981528043747, 0.6193698644638062, -0.3387141227722168, -0.8977130651473999, -0.27395856380462646, 0.3352862298488617, 1.5940264463424683, -0.49547678232192993, -2.1256911754608154, 1.0358386039733887, 0.022947225719690323, 0.6203240156173706, -0.08439626544713974, -0.015950627624988556, -1.899071455001831, 0.5681025981903076, -1.0902847051620483, 0.6460975408554077, -1.9031356573104858, 1.1568630933761597, -0.19431565701961517, -2.3191583156585693, 0.08533912152051926, 0.061499934643507004], [0.06718435883522034, -1.1371179819107056, 2.1765944957733154, -0.5399312376976013, 0.9490607976913452, -2.5173676013946533, 0.565463125705719, 0.4415055513381958, -0.06854431331157684, 1.1811407804489136, -0.6423947811126709, 0.40023279190063477, -0.11146780848503113, 0.22484862804412842, -1.0148693323135376, 0.5947418212890625, 0.5010231733322144, 0.2533967196941376, 0.2678269147872925, -0.4248562157154083, -0.6356867551803589, -0.25460144877433777, 0.3373798131942749, -1.2296785116195679, -0.1287633329629898, 1.5118653774261475, 0.6943825483322144, -0.9484513401985168, 1.206198811531067, -1.6774423122406006, -0.21317793428897858, 1.089855670928955, 0.6535452604293823, 0.3427998721599579, -1.0250674486160278, -1.623550295829773, 0.5153569579124451], [0.2664427161216736, 1.6524529457092285, -0.5870410799980164, -1.1012121438980103, -0.5653835535049438, -1.8360902070999146, -0.3958655893802643, -0.10237638652324677, -0.223006933927536, 1.726439356803894, 1.7942090034484863, -0.37923410534858704, -0.5978354215621948, 1.211462378501892, 1.0117062330245972, -1.2673676013946533, -0.7685150504112244, -0.8158620595932007, -1.3179686069488525, -0.6077525615692139, -1.2879129648208618, 0.7033165097236633, 0.4363970160484314, -1.7370073795318604, -0.6779181361198425, -0.6267222166061401, 1.3483995199203491, -0.6731978058815002, 0.8325881361961365, 1.5741504430770874, -0.9007415771484375, 0.5036454796791077, 0.08004190772771835, 0.8664320111274719, -1.3726229667663574, -0.3784315288066864, -1.6599268913269043], [0.7102357149124146, -1.9658007621765137, 0.06089039146900177, -1.5742381811141968, -0.7501658201217651, -0.828077495098114, 0.06179714947938919, 0.5364381074905396, 1.141093134880066, 1.3673439025878906, 0.03442516550421715, -1.0948697328567505, 1.23318350315094, -1.231382131576538, 0.01232623215764761, -0.6832202076911926, -0.606899082660675, 0.2018967866897583, -1.1232601404190063, -0.08511637151241302, -0.08092609792947769, 1.0655921697616577, 1.1429513692855835, 0.794396698474884, 0.07386167347431183, 1.7457424402236938, 1.00190007686615, -1.2347458600997925, 0.3332694172859192, 0.4916214644908905, -0.8077965974807739, 0.27732980251312256, -0.293396532535553, -0.8676804900169373, 0.08948227763175964, -0.5941113233566284, -1.195180058479309], [0.04651811718940735, -1.8242813348770142, 1.1651135683059692, -0.31754744052886963, 0.11684279888868332, 0.9931266903877258, 0.969288170337677, -0.2675635516643524, 0.3591177761554718, 0.9989314675331116, 1.4412182569503784, -0.7639109492301941, 0.32898449897766113, -1.061725378036499, -0.8728103041648865, -0.0621672123670578, -0.19894680380821228, -1.085396409034729, -0.90152907371521, -0.0826646015048027, -1.4480808973312378, 0.14759361743927002, -1.209708571434021, 1.3939419984817505, 0.1551164984703064, 0.7218945026397705, -0.39178726077079773, 1.0853086709976196, 0.48951900005340576, 0.7251925468444824, 0.08792734146118164, 0.6050485968589783, -0.12375199794769287, 0.5931665897369385, -0.32291513681411743, -0.33407047390937805, 0.13778150081634521], [0.08681734651327133, -0.07236835360527039, -0.7953818440437317, -0.281545490026474, 0.9269010424613953, 0.11631805449724197, 0.3234294652938843, -0.5294961333274841, 1.38230299949646, -0.2191593497991562, 0.5384920239448547, -0.0319056361913681, 1.192252516746521, -1.5924581289291382, -2.0901620388031006, -0.37041670083999634, -2.54483962059021, -0.8309440016746521, -0.7712511420249939, -0.3129863739013672, 0.6078916192054749, -1.411118507385254, 0.4761665165424347, 1.2079243659973145, 1.5795178413391113, -1.1589618921279907, -0.39497628808021545, 1.8992760181427002, -1.0264208316802979, 1.1277997493743896, 0.3625451326370239, 0.5550692081451416, -1.6864936351776123, -1.4365801811218262, -0.8995394110679626, 0.6149368286132812, 0.8047661185264587], [-0.808057427406311, 0.2906768321990967, 0.13196377456188202, -0.981905996799469, -1.1221961975097656, -1.9851257801055908, -1.191908359527588, -0.8790076971054077, 1.172463059425354, 0.7775416970252991, -0.27158477902412415, 0.2406950145959854, -0.9363154768943787, -2.446807861328125, 0.15612927079200745, 0.4737427234649658, -0.8897286653518677, 1.1089973449707031, 0.9548251032829285, 0.7152148485183716, -2.0058670043945312, -0.06099752336740494, 0.5299941301345825, -0.9827827215194702, -0.6286085844039917, 0.7582128047943115, 1.9748839139938354, 0.7290029525756836, 0.15627939999103546, -0.5930910110473633, -1.172780156135559, -0.9369434118270874, 0.30520325899124146, -0.5651227831840515, 1.2858343124389648, -1.409050703048706, -1.2209584712982178], [-0.1064319834113121, 0.1698094755411148, 0.0731416642665863, -1.1270489692687988, -0.773586094379425, 0.9273741841316223, -0.3940889239311218, -0.4976324141025543, 0.6356099247932434, 0.40446794033050537, 0.011896487325429916, 1.6982038021087646, -1.3584794998168945, 0.3872000277042389, -0.5964857339859009, 1.0117472410202026, -0.9571547508239746, 0.6676332354545593, -0.39243194460868835, 2.552928924560547, 0.24354447424411774, 2.227494239807129, -0.6650217175483704, -0.5956724882125854, 0.36941421031951904, -1.7032631635665894, -0.9310604333877563, 1.5138170719146729, 1.1193768978118896, -0.20753945410251617, 1.135641098022461, 0.4718017280101776, 0.48492997884750366, -0.9475804567337036, 0.981362521648407, -0.5574548840522766, 1.5072251558303833], [-1.0213136672973633, 2.5242574214935303, -0.651451587677002, -1.3515305519104004, -1.1158841848373413, 0.396743506193161, 0.2996715009212494, -0.6765679717063904, 0.7828352451324463, 1.9653102159500122, 0.14229068160057068, 0.7915492653846741, 0.907976508140564, 0.8754189014434814, -1.7130459547042847, 1.5439622402191162, -0.004590985830873251, 0.18295887112617493, 1.2606382369995117, -2.6089508533477783, 0.8699716329574585, 0.35738512873649597, 1.7365580797195435, 0.4444422721862793, 1.4298498630523682, 1.161872386932373, -1.1219216585159302, 0.014488314278423786, -0.6531149744987488, 0.004985859151929617, -0.8151426911354065, -1.4523301124572754, -0.06815265119075775, -0.2651767134666443, 0.5280606150627136, 1.7144854068756104, -0.3140673041343689], [-0.17046207189559937, 1.2715842723846436, 1.024749994277954, -1.5545328855514526, 0.01023827400058508, -1.4930379390716553, -0.8514704704284668, -0.50489342212677, -0.16574513912200928, -0.31808599829673767, 0.17840217053890228, 0.2947404384613037, 1.5483617782592773, 0.7665497064590454, -1.0987792015075684, 1.3248357772827148, 0.28460150957107544, -1.2050336599349976, -1.0193684101104736, 0.010982408188283443, 0.058362819254398346, 0.3085726797580719, -1.5636368989944458, -0.37450528144836426, 2.0377650260925293, -1.7772551774978638, -1.3940297365188599, -0.016868114471435547, 0.24004629254341125, -0.632236123085022, 0.09142911434173584, -0.1670985072851181, -0.9067343473434448, -0.09956006705760956, -0.3803108036518097, 0.2822422385215759, 0.6216264367103577], [-0.956899881362915, -0.16766215860843658, -1.1195728778839111, -0.4274812340736389, 0.05756540969014168, 0.6331838965415955, -0.8580659627914429, 0.5600736737251282, -0.16562338173389435, -0.32980677485466003, -0.3024408519268036, -0.6646692752838135, 0.2840230166912079, -1.313971757888794, 0.3831874430179596, -0.6067637801170349, 1.387538194656372, -0.5688409209251404, 0.2432083785533905, 0.6464277505874634, 0.7004852294921875, 1.487210750579834, 1.58077871799469, -0.18564023077487946, -0.4956205189228058, 1.3872056007385254, 0.39653173089027405, -0.11778825521469116, 0.026715589687228203, 0.8687761425971985, 0.8127901554107666, -2.097456693649292, -0.7656363844871521, -0.5273882746696472, -0.9985038042068481, 1.137657642364502, 1.2447339296340942], [-0.10537359863519669, 1.450626015663147, -0.2625804841518402, 1.427547812461853, 0.8620073795318604, 0.31437548995018005, 0.6343536972999573, -0.3676559031009674, 1.256301760673523, -0.22845420241355896, 0.7125592231750488, -0.6422973275184631, -0.6058006286621094, 0.29251885414123535, 0.32115212082862854, 1.4461389780044556, -0.00675328029319644, 0.5160676836967468, 0.5581396222114563, 2.1584486961364746, 0.3301209509372711, -0.9470863342285156, -1.3016880750656128, -0.26005321741104126, 0.16846579313278198, -1.0806654691696167, -1.1014260053634644, -0.9092336297035217, -0.017242124304175377, 0.5936139225959778, -0.7265195250511169, -0.8437180519104004, 1.2214617729187012, 1.7023496627807617, 0.4950777292251587, -0.6162407398223877, -0.6309025287628174], [0.6297774910926819, -1.7826452255249023, -1.6500009298324585, 0.8484422564506531, -1.5137839317321777, -0.8620281219482422, 1.9465134143829346, -0.5832497477531433, 1.2387006282806396, -0.3740774095058441, -1.1704868078231812, 0.3451206386089325, -0.1872887760400772, 0.7458148002624512, -1.535520315170288, 0.4385029673576355, 0.16173848509788513, 0.7393901944160461, 0.21805191040039062, -1.0146222114562988, 1.275256872177124, 0.86155104637146, 0.883017897605896, 0.578018307685852, 0.32909056544303894, -0.5924285650253296, -1.7864434719085693, -1.5859148502349854, -0.19567179679870605, -2.2688989639282227, -0.6854172945022583, 0.7853512763977051, -0.985656201839447, 1.8164960145950317, -1.7258974313735962, 0.5975903272628784, -2.000681161880493], [0.2623961865901947, 1.5673105716705322, -0.39304667711257935, -1.3177021741867065, -1.2105103731155396, 0.5641721487045288, 0.7709922194480896, 0.3180501461029053, -0.5852661728858948, -1.171015739440918, -0.9235130548477173, -1.1333928108215332, -1.6841168403625488, 0.954544186592102, 1.1546199321746826, 1.1951990127563477, 1.6044511795043945, -0.7596399188041687, -1.9524993896484375, 0.10079453885555267, -0.3881426453590393, -1.2428784370422363, 0.3050044775009155, -0.5764957070350647, 0.5294331908226013, 0.26749834418296814, -0.979400098323822, -0.1414753794670105, -0.6667485237121582, 0.19186192750930786, -0.2122999131679535, -1.116409420967102, 0.26757219433784485, -0.38260599970817566, -0.3154464662075043, -0.04246320202946663, 2.049114942550659], [0.2291177660226822, -0.35141828656196594, -0.7904560565948486, -1.8020095825195312, -0.16896562278270721, -0.13699869811534882, -0.6979727149009705, 0.5829333662986755, 0.2599465250968933, 0.5254612565040588, -0.3848252296447754, 1.364610195159912, 0.45751506090164185, -0.7480149269104004, -0.10653332620859146, 0.8824681043624878, -0.7350720763206482, 1.1581757068634033, -0.5183622241020203, -1.904365062713623, 0.6088518500328064, -0.6085742712020874, -0.06473804265260696, -0.25303956866264343, 0.9509009122848511, -0.1136271208524704, 0.7477946877479553, -0.9657898545265198, -0.04744137451052666, -0.7023696899414062, -0.12603262066841125, 2.0281121730804443, -0.8876428604125977, 0.2549999952316284, -0.4704214632511139, 0.29164233803749084, -0.9608997106552124], [-0.6758156418800354, -1.8347903490066528, 1.0805773735046387, -0.08651623874902725, 0.11148428916931152, -1.4319820404052734, 1.3305156230926514, 0.947045087814331, -0.5020083785057068, 0.13058310747146606, -0.4056552052497864, 0.7896215319633484, -2.2955856323242188, 0.0474541112780571, 0.11638076603412628, 0.7028399109840393, -1.1435704231262207, -0.868742823600769, -1.180406928062439, 0.7930412888526917, 0.2955884039402008, 0.9430834054946899, 0.7037307620048523, 0.5577419996261597, -1.2189334630966187, -0.27355894446372986, 1.2401784658432007, 0.2802971303462982, -1.0042730569839478, -0.8722139000892639, -0.47487205266952515, -0.2715705633163452, 0.08444838970899582, -0.013329911977052689, 0.4245009124279022, 1.0125101804733276, 0.6059505343437195], [-1.0221030712127686, 1.9986989498138428, -0.22209851443767548, -0.6624348759651184, 1.076045274734497, 1.3673254251480103, 0.4824461340904236, 1.2884995937347412, -0.14496387541294098, -0.11891043931245804, -1.050093412399292, 1.0446349382400513, 0.23107850551605225, 1.6607820987701416, -0.25369182229042053, 1.1669126749038696, 0.17555277049541473, 1.7337783575057983, 0.10593368858098984, 1.476641297340393, -0.8935574889183044, -0.31003886461257935, -0.41529473662376404, 1.12408447265625, -1.941065788269043, -1.034184455871582, 0.3089647591114044, 0.04182210564613342, 0.0489327609539032, 0.5021927356719971, -0.13999439775943756, 0.7252249121665955, 1.3374145030975342, -0.8322656154632568, 1.3010072708129883, -0.6911340355873108, -1.2246121168136597], [-0.037877969443798065, -0.07954960316419601, 0.28730878233909607, -1.1041992902755737, -0.09762663394212723, -0.5815989971160889, 1.042155385017395, 0.03782827407121658, -0.6252031922340393, -1.0425184965133667, -0.045329462736845016, 1.8599200248718262, -1.710795283317566, -0.6925480365753174, 1.16171133518219, 0.954355001449585, -0.24005375802516937, -0.15857675671577454, -0.45675334334373474, -0.8952217698097229, -0.310628741979599, 0.49065449833869934, 0.33204418420791626, -1.4736995697021484, -0.21458126604557037, -0.6569597125053406, 1.1781920194625854, -1.3504990339279175, -1.135765552520752, -1.3258116245269775, 3.470853567123413, 1.7114919424057007, -0.20218917727470398, 0.9649333953857422, -1.67161226272583, -0.21061940491199493, 2.280519723892212], [-0.13390983641147614, -1.2822877168655396, -0.5957999229431152, -0.5719497203826904, -0.14877407252788544, -0.14393006265163422, -0.4750649333000183, -1.47385573387146, -0.0382615365087986, -0.027571210637688637, -1.573366403579712, 0.5500587821006775, -0.167697012424469, 1.7047260999679565, 0.4825766682624817, -0.633487343788147, -0.8705427646636963, 1.7674317359924316, 1.1736541986465454, 0.5679895281791687, 0.8726391792297363, -1.3030147552490234, 1.19464910030365, -0.08994323015213013, 0.019594311714172363, 1.0780844688415527, 0.6226696372032166, -1.4356218576431274, 0.7344527244567871, -0.08710635453462601, 1.252476692199707, 0.28521955013275146, 0.7164965867996216, -0.8297839760780334, -0.6872748732566833, -0.9597731232643127, 0.4750637114048004], [-0.7537443041801453, -0.14862120151519775, 2.3316519260406494, 1.1387616395950317, 2.672050714492798, -0.3981418311595917, -0.531471312046051, -0.08350002020597458, 0.9941527247428894, -0.4553574025630951, 2.3642516136169434, -0.5091432332992554, 1.7357988357543945, -0.6363602876663208, -0.4804753363132477, 0.7009579539299011, 0.30478689074516296, -0.01367251854389906, -0.15346062183380127, 0.5751044750213623, -0.2806470990180969, 2.1240830421447754, 1.0440821647644043, 0.7617460489273071, -0.7317972183227539, 3.0672969818115234, 0.5401474237442017, -0.45339062809944153, -0.717154860496521, -2.33591890335083, -0.48279985785484314, -1.890964150428772, -1.363298773765564, 1.4097402095794678, -0.28417715430259705, -0.7724227905273438, 1.7372164726257324], [-0.6137104630470276, 2.080686092376709, -0.2398786097764969, 0.4297613501548767, -0.33099958300590515, -0.5147132873535156, 0.255504846572876, 1.4222151041030884, 0.1504323035478592, -0.9414344429969788, -1.6860251426696777, 1.2663460969924927, -0.6581678986549377, -1.0113673210144043, 0.5297312140464783, 0.5487570762634277, 0.6631815433502197, -0.5483876466751099, -1.8369429111480713, 0.6205257773399353, 0.022571148350834846, 0.007791006006300449, -0.8434587717056274, 0.5930455327033997, 0.11558539420366287, 0.33820056915283203, 1.280469536781311, -0.1837526261806488, -0.2921803295612335, -0.2797658145427704, -1.573616862297058, -0.3598836362361908, 0.9950661063194275, 0.7856020331382751, -0.5653690695762634, -1.2304250001907349, 0.43211397528648376], [0.019508248195052147, 0.13425183296203613, 1.4544225931167603, -0.24484652280807495, 1.0872470140457153, 0.2773854732513428, 0.8206560015678406, -0.7318146824836731, -0.09896150231361389, 0.27744776010513306, -0.18117806315422058, 0.04001687467098236, -0.31156548857688904, 1.7707476615905762, -2.270023822784424, 0.7756788730621338, 0.07296585291624069, -1.1631096601486206, -0.6562923789024353, -1.5232914686203003, 0.5768181681632996, 0.714580774307251, 0.39870357513427734, 1.6264110803604126, -1.705030918121338, -1.3068498373031616, -0.46672841906547546, 0.693999171257019, 0.44435882568359375, -1.085906982421875, -0.001034239074215293, 0.7417160868644714, -1.504495620727539, 0.5227938294410706, 0.8853649497032166, 1.0405516624450684, 1.0181759595870972], [-0.3610225319862366, -2.1780548095703125, -1.4462093114852905, -0.9693025946617126, -0.3147036135196686, 0.6716954708099365, -0.03507595509290695, 0.2332453578710556, 1.3278175592422485, -1.6428842544555664, 0.18885521590709686, 0.5993847846984863, 1.24256432056427, 2.158290386199951, 0.9946760535240173, -1.1087387800216675, 0.5447980761528015, -0.8969888091087341, 0.38274556398391724, -0.6626460552215576, -0.5653402805328369, 0.23726339638233185, -0.660043478012085, 0.04336380213499069, -0.09391043335199356, 0.49780818819999695, -0.6290435194969177, -1.1961379051208496, -1.8147519826889038, 0.4314321279525757, 0.6003120541572571, -1.0913589000701904, -1.9080922603607178, 0.6532422304153442, -0.0003821424033958465, 0.7583563327789307, 0.007645103614777327], [1.236222267150879, 0.362144410610199, -0.09535302966833115, 0.9628497958183289, -1.1421825885772705, -0.06726748496294022, 0.09778045862913132, -0.33013808727264404, 0.3040611445903778, -0.42219284176826477, -1.6820544004440308, 1.9216551780700684, -0.039506494998931885, -0.29462599754333496, -1.436145544052124, 0.015916328877210617, -0.21741490066051483, -1.4335500001907349, 1.146555781364441, 0.9228423237800598, -1.8368784189224243, 0.7644543647766113, -0.7550212740898132, 0.7733451724052429, 0.5207905769348145, 1.023114800453186, 0.567315399646759, 1.4828901290893555, -0.31084007024765015, -0.3151382803916931, -0.2976972758769989, -0.4105280935764313, 2.184776782989502, 0.20220991969108582, 1.3065377473831177, 0.2908689081668854, -0.043977782130241394], [0.975932240486145, -0.5972358584403992, 0.7211387753486633, 1.327086091041565, 0.15812110900878906, 0.217146098613739, -0.33700764179229736, 0.7488210797309875, 0.9971998333930969, 1.1235151290893555, -0.33767181634902954, 0.08226113766431808, -0.7797991633415222, 0.4405433237552643, 0.7477113008499146, 0.17916500568389893, -0.04404586926102638, 0.4347020089626312, 0.3148775100708008, 0.49300891160964966, -0.6696890592575073, -0.8021634221076965, -0.46481651067733765, -0.8732020258903503, 0.314391165971756, 1.9869463443756104, 0.10058432072401047, -1.6047141551971436, -0.7842961549758911, -0.9773681163787842, 0.48794811964035034, 0.21754351258277893, 1.7411285638809204, 0.09903404861688614, -0.14708992838859558, 1.2357677221298218, -0.9419130682945251], [-0.025824852287769318, 1.1267236471176147, 0.24108141660690308, 0.4148668944835663, -0.9126825928688049, -1.524875283241272, -0.302156925201416, 0.9810658693313599, -0.5491502285003662, -1.074233889579773, -1.5399765968322754, 1.8609362840652466, -1.5578161478042603, -0.6197042465209961, -1.156915307044983, 0.6645376682281494, -1.6868056058883667, -1.401363730430603, 0.19311828911304474, -1.6058051586151123, 0.5066274404525757, 1.1662461757659912, 0.10733872652053833, -2.5208353996276855, -0.6470345258712769, -0.2534470856189728, 1.3290292024612427, 1.5141116380691528, 1.0074446201324463, 0.05849859118461609, -0.2289838045835495, -0.09306085109710693, -0.6182472705841064, -0.4326978027820587, -0.5804596543312073, -0.01840975135564804, 1.2792949676513672], [-1.3333746194839478, 1.8580987453460693, 0.6271287798881531, 0.9071940779685974, -0.024268537759780884, -0.04621797055006027, 1.2324694395065308, -0.380130797624588, -0.9049882292747498, -0.04597313329577446, -0.5538161993026733, 1.358750581741333, 0.1731918603181839, 0.5628251433372498, 0.0923745408654213, 0.03775840625166893, 1.1995879411697388, 0.36671292781829834, 1.5397287607192993, -1.7581614255905151, 0.5138025283813477, 0.9563080668449402, 0.01514147873967886, -1.7128559350967407, 0.04635733366012573, -0.20937874913215637, 1.3682348728179932, 1.2526698112487793, 1.144927978515625, 0.3444370925426483, 0.45943892002105713, 0.37439751625061035, 1.2204822301864624, 0.20700733363628387, 0.8204845786094666, 1.0660221576690674, 0.1284959316253662], [0.38948866724967957, -0.44364070892333984, -0.8778297305107117, -0.4086071252822876, -0.519166886806488, -1.3395280838012695, 0.33162614703178406, 1.0241037607192993, 1.0836560726165771, 1.6773778200149536, -0.04353271797299385, 2.410490036010742, -0.15888100862503052, 0.3985559642314911, 1.514561653137207, -1.4141709804534912, 0.5849094986915588, 0.5466939806938171, 2.3135671615600586, -0.4086609482765198, 0.36172759532928467, -2.241137742996216, 0.17328579723834991, -1.3884204626083374, -1.1047910451889038, 0.7710968255996704, -0.5128144025802612, -1.2006644010543823, 0.8499035835266113, 0.26294299960136414, -0.301605224609375, -1.4507339000701904, -0.6479091048240662, -1.2442584037780762, -0.2234470546245575, -0.30555862188339233, -0.08519618213176727], [-0.628454327583313, -0.2554247975349426, 0.9196680188179016, -0.4717458188533783, 0.895895779132843, 0.7956743836402893, 0.7759091854095459, -1.7451541423797607, 0.8006926774978638, 0.13579988479614258, 0.4472099840641022, -0.2615218162536621, -0.2751101851463318, -1.1106853485107422, -1.2218148708343506, 0.28107985854148865, -0.9133523106575012, -0.5126992464065552, 0.8950684666633606, 1.2795382738113403, -0.15122564136981964, -1.992605447769165, 0.3149578273296356, 0.9229974746704102, 0.0752662643790245, -0.35362324118614197, -1.0949575901031494, 0.19160610437393188, 2.7606799602508545, 0.6658317446708679, -0.2850112020969391, -0.8633749485015869, 0.48494818806648254, -1.3167407512664795, -0.46376749873161316, -0.19925573468208313, -0.21852397918701172], [0.46323591470718384, 0.004755478817969561, -1.1448206901550293, -2.041618824005127, 0.16123640537261963, -2.6243948936462402, -1.3668030500411987, 2.080935001373291, -0.712727963924408, 1.4559967517852783, -0.6169538497924805, 0.3719404935836792, 0.7074043154716492, 1.1648650169372559, -0.2602837383747101, -1.126435399055481, -0.00047471412108279765, -1.5023781061172485, -1.4749948978424072, 1.0089845657348633, -0.8466166853904724, 1.2305386066436768, 1.1766109466552734, -0.7916373014450073, -0.4062943458557129, -0.20417667925357819, 1.0637602806091309, 0.0690372884273529, 1.2536051273345947, 0.061940111219882965, 1.2372450828552246, -0.07188931107521057, 1.4519457817077637, 0.9747440814971924, 1.4963303804397583, 1.139414668083191, -1.5061408281326294], [2.041245222091675, 0.30574148893356323, -1.263829231262207, -0.4268903136253357, -0.7614427804946899, 0.7909106612205505, -0.1997040957212448, 0.1562904268503189, 1.1134172677993774, -1.3759747743606567, -1.3379182815551758, 0.31536415219306946, -1.4035322666168213, 0.94057697057724, -1.0726196765899658, -0.888465940952301, 0.8556349277496338, 1.6261980533599854, 0.026605412364006042, -0.38400998711586, -1.3386677503585815, -0.3040359318256378, -0.410880982875824, -1.0741652250289917, -0.45450156927108765, 0.8904129862785339, 0.13872568309307098, -2.514625072479248, 1.648657202720642, 1.126834511756897, -1.231253743171692, 1.2509726285934448, -1.1651015281677246, 0.5812910199165344, 0.18629787862300873, -0.9218732118606567, 0.3855212330818176], [0.1753179430961609, 0.2315252721309662, -1.241953730583191, 0.7917213439941406, -0.20479795336723328, 1.3990867137908936, 0.888725221157074, 0.7276357412338257, -1.0259181261062622, 2.019092559814453, 0.4125823974609375, 0.02999996580183506, -0.5118309855461121, 0.0261305533349514, 1.1440249681472778, -1.9724844694137573, 0.5233870148658752, -1.1966406106948853, -0.8465293645858765, 0.505220353603363, 0.5757696032524109, -0.19494198262691498, 0.3903840482234955, 1.5434350967407227, 0.1734379231929779, -1.017199158668518, -0.27111807465553284, -0.3040958046913147, 1.4674041271209717, -0.3117809593677521, -1.6240602731704712, -1.2797962427139282, 1.2565631866455078, 0.5457111597061157, -1.0757057666778564, -1.3709499835968018, 0.806137204170227], [-1.2285994291305542, -0.20532459020614624, 0.8122590780258179, 0.6603960990905762, -0.9328356981277466, 0.5419020652770996, -1.5941652059555054, 0.23313038051128387, -0.604525625705719, -1.912650465965271, 1.2335046529769897, -0.2472343146800995, 0.4073115885257721, -0.48495587706565857, 2.7985494136810303, 1.5328724384307861, -0.2310149073600769, 1.9776651859283447, 0.9175072908401489, 1.6315051317214966, -2.0321996212005615, 0.29390662908554077, -1.0836676359176636, -0.14750176668167114, -0.3711611032485962, 0.2616468667984009, -2.0250325202941895, -0.381315678358078, -1.4664345979690552, 1.0272774696350098, 0.25317129492759705, 0.6089096665382385, 0.9896445274353027, 0.7418103218078613, -0.6519814729690552, 0.05609453096985817, -1.901872992515564], [-0.4851335883140564, -0.4119527041912079, -0.2782365679740906, 0.30178102850914, -0.028323428705334663, 1.7179347276687622, 0.9498697519302368, 0.061458151787519455, -0.28183767199516296, -0.5198541283607483, 1.218617558479309, -0.7504199147224426, 0.9296014904975891, -1.0866178274154663, -0.07822712510824203, -1.2658376693725586, 0.4676203727722168, 0.22136420011520386, -0.4841712415218353, -1.5880284309387207, -2.7196202278137207, -0.5606428980827332, 1.0015897750854492, -0.974854588508606, 0.2148425430059433, -0.9587894082069397, 0.6007398962974548, -0.9489567279815674, 1.2466729879379272, -0.9317299723625183, -0.31450146436691284, 1.689524531364441, 0.27851107716560364, 0.09687934070825577, -0.13046270608901978, 0.42403358221054077, -0.9782085418701172], [-1.105452060699463, 1.4983505010604858, 0.02032223902642727, 0.23323790729045868, 0.004585507791489363, 0.14297308027744293, 1.202733039855957, 0.2313079982995987, -0.07963425666093826, -0.5842060446739197, 0.21952836215496063, 1.1727675199508667, -0.48098909854888916, -0.3765714466571808, 1.2440718412399292, 0.2824053466320038, 1.158592939376831, -0.9728068709373474, -0.6727427840232849, -0.9680476784706116, 0.48746830224990845, -1.4394352436065674, 0.6089777946472168, 0.037381719797849655, 0.5756974816322327, 0.9090427160263062, 1.0571019649505615, -2.245194435119629, 0.7135568857192993, 1.21070396900177, -0.2920069396495819, 0.7915632128715515, 0.7462379932403564, 0.8579650521278381, -0.9578611850738525, 1.4355560541152954, -1.171035647392273], [0.6724768280982971, 0.8123719096183777, 0.748815655708313, 0.6216731071472168, 1.7165194749832153, 0.9321103096008301, 0.6318390369415283, 1.4200106859207153, -1.1659808158874512, -0.19060125946998596, -0.9581177234649658, -1.0621198415756226, 1.1943213939666748, 0.3546817898750305, 1.7141432762145996, 0.8489084243774414, -2.8262317180633545, -0.21885447204113007, -0.37271392345428467, -0.5806688666343689, 1.7639994621276855, -0.4920605719089508, 1.1314287185668945, -0.049344711005687714, -0.07917346805334091, -0.5662944912910461, 1.7668319940567017, 0.01407957449555397, -0.9908146262168884, 0.19281131029129028, -1.404628038406372, 1.3623683452606201, -1.1712760925292969, -0.6469005346298218, 0.24965453147888184, -0.992835521697998, 1.4992927312850952], [-0.0031018108129501343, 0.14094427227973938, 0.10731545090675354, 0.020080214366316795, 0.15355539321899414, 2.0044350624084473, -0.19077710807323456, -0.9643410444259644, 1.3069347143173218, 1.0174132585525513, -0.3952852487564087, -0.4035082757472992, 1.3340709209442139, 0.19814784824848175, -0.017136698588728905, -0.1019316166639328, -0.6127792596817017, 0.655373215675354, -1.3434088230133057, -1.1411287784576416, -1.795488953590393, -0.597737193107605, -1.1265404224395752, 0.27186357975006104, -0.7303130030632019, -0.18772931396961212, 0.6480793356895447, 0.06927783787250519, -0.6631335616111755, -0.5331347584724426, -0.7556295990943909, 0.05295165628194809, 0.10122773796319962, 1.3541693687438965, 0.7716454863548279, 0.4015183746814728, -0.5819754600524902], [0.374470591545105, -1.0144182443618774, -1.892857551574707, 0.5025742650032043, 0.8507146835327148, -3.225766897201538, 0.977071225643158, -1.6976782083511353, -1.0872656106948853, 0.41360169649124146, -0.2729218602180481, -0.12868019938468933, -1.1546117067337036, 1.1920387744903564, -0.2092157006263733, -0.278315007686615, 1.613572359085083, -0.023025162518024445, 0.5684941411018372, 0.7906660437583923, -0.0013321563601493835, -1.3282554149627686, 0.5924720168113708, -0.6653545498847961, 1.331194519996643, 1.1413781642913818, -3.0804860591888428, 2.0534451007843018, -1.270135521888733, 0.26979389786720276, 0.43478280305862427, -0.048348840326070786, 0.3756307065486908, 1.3686797618865967, 0.34878817200660706, -0.1367291361093521, 0.1459624320268631], [-0.0012884772149845958, -0.10761252045631409, -0.007802044041454792, -0.07804252952337265, 1.0076597929000854, -0.5576692223548889, -0.9738683700561523, 1.4340288639068604, -0.10085263848304749, 1.4616667032241821, 0.2964564859867096, -0.697716236114502, 0.6065371632575989, 2.092595100402832, -0.592175304889679, -0.1491454392671585, -1.355175256729126, 0.46480581164360046, 0.059184614568948746, 2.3919098377227783, -2.0422556400299072, 0.704399824142456, 0.292635977268219, 0.7888522148132324, -0.05014243721961975, 0.429840624332428, -0.3749217987060547, -0.21993592381477356, 1.0371228456497192, -2.310858964920044, 1.2984663248062134, -0.5275577902793884, 0.12235726416110992, 1.3713724613189697, 1.3670949935913086, 0.3753093183040619, 0.10874774307012558], [-1.3316842317581177, 0.14936988055706024, -0.1639552116394043, -0.4236403703689575, -1.0144692659378052, 0.00026421304210089147, 0.5514443516731262, -0.60782390832901, -1.597217321395874, 0.5168174505233765, -1.051795244216919, -0.6882209181785583, -0.3972183167934418, -0.5499991178512573, -1.1608104705810547, -0.9011721014976501, -0.16741827130317688, -0.6081509590148926, 0.031188156455755234, 0.6933453679084778, 1.0177628993988037, -1.135606050491333, -0.7555854320526123, -0.4367387294769287, -0.3996821343898773, 0.21739718317985535, -0.09529072046279907, -1.9301581382751465, -1.7229487895965576, 0.09277752041816711, 0.9403852820396423, 1.004665732383728, 0.26179835200309753, 0.21726961433887482, -1.4620949029922485, -1.676228642463684, -0.24322907626628876], [2.6296591758728027, 0.9052867889404297, -0.5385928750038147, -0.8628060221672058, -0.6666383743286133, -0.8322064280509949, 1.1777279376983643, -1.1950501203536987, 0.38141584396362305, -0.7069892883300781, 0.19690357148647308, 0.9911648631095886, -0.01821824535727501, 0.0355270616710186, -0.8720816969871521, -0.7476170659065247, 1.5645993947982788, -0.6437221169471741, -0.19904275238513947, -1.2056002616882324, 0.3973890244960785, -0.9482331275939941, 0.21369874477386475, -0.4366488456726074, -0.7461897730827332, 0.3256348967552185, 0.7522240877151489, -1.156474232673645, 0.9512456059455872, 0.1512216031551361, 0.7873844504356384, 1.0460273027420044, -0.5425755381584167, -0.30094006657600403, 0.2193416953086853, 1.0500342845916748, -2.3821706771850586], [1.59840989112854, -0.27429988980293274, -1.7042748928070068, -0.22311507165431976, -1.0563615560531616, 1.8342732191085815, -0.4958428144454956, -0.6602951288223267, -0.43094781041145325, -0.984502911567688, 0.08851730078458786, 0.5072659850120544, 0.0682770311832428, -0.9651879072189331, 0.805996298789978, -0.19520510733127594, -0.3202724754810333, 0.9849141836166382, 0.0897202342748642, -2.8114171028137207, 0.46390244364738464, -1.943295955657959, -1.2598057985305786, -0.44899001717567444, 0.06732764095067978, 0.25130051374435425, -0.5036259889602661, 0.6892284750938416, 0.31042975187301636, -0.40835535526275635, -1.686637043952942, 0.3857918977737427, -1.289442539215088, -0.9307626485824585, -0.6582608819007874, -0.7863543629646301, -2.369952917098999], [-0.19782422482967377, -1.0685644149780273, -0.8584596514701843, 0.5849199295043945, 1.6115700006484985, 0.12354446202516556, -0.8814509510993958, -0.7375573515892029, 1.0347974300384521, -0.10526077449321747, -1.2580549716949463, 2.1284189224243164, 0.7099142074584961, -0.7241125702857971, -0.7328348159790039, -1.6691696643829346, -2.1948845386505127, 0.052691660821437836, -0.9543310403823853, 1.2262088060379028, -0.7346204519271851, -0.21381404995918274, 0.7556650042533875, -1.4991121292114258, 0.4032646715641022, -1.7259461879730225, 2.314210891723633, -1.4216392040252686, -0.952990710735321, 2.378568649291992, -0.5368362665176392, 0.6481872200965881, -0.1428719013929367, -2.313368797302246, 0.03342462703585625, -1.0268687009811401, 1.6956846714019775], [-1.1580636501312256, 1.6609317064285278, -0.43921738862991333, 0.3436649739742279, 2.5101029872894287, -0.4768388569355011, -0.45339465141296387, 0.5710642337799072, -1.1129151582717896, 0.7729710936546326, 0.7998595237731934, -0.061346542090177536, -1.8747401237487793, -2.4058127403259277, -0.9396057724952698, -0.5511478185653687, -2.4382286071777344, -0.3743763864040375, -0.04068952426314354, -0.05604340136051178, 1.1928035020828247, -0.5926553606987, 0.21121032536029816, -0.934700071811676, 0.9046950936317444, -0.35064226388931274, -0.3029082715511322, -0.006793163251131773, -2.149122714996338, 0.8448337316513062, -0.7045511603355408, -0.31872037053108215, -0.2829122245311737, 2.560123920440674, -0.0843127965927124, -0.6701292395591736, -0.17662958800792694], [-0.34963351488113403, -0.38572296500205994, 0.09988816827535629, 0.9804601073265076, -1.1925405263900757, 0.3444429934024811, -0.20197917520999908, 0.03223789855837822, 0.9173483848571777, -0.21356748044490814, -2.651007890701294, 0.633892297744751, 0.6742017865180969, 0.6409488320350647, -0.748755156993866, 1.692423701286316, -0.22747062146663666, -0.8863959312438965, 1.901077389717102, -2.566270351409912, 0.6342580914497375, 0.49740472435951233, -0.37799569964408875, -0.8238567113876343, 1.500766396522522, 1.2215133905410767, 1.676836371421814, 0.07750412821769714, -0.04011260345578194, 0.037983741611242294, -1.2884610891342163, 1.4033631086349487, 1.2814651727676392, 0.8382388353347778, -1.328931212425232, -0.1121068075299263, 0.44283193349838257], [0.6067622303962708, -0.6922342777252197, 0.7589086294174194, 1.7692407369613647, -1.2084436416625977, -0.4196876585483551, -1.3609541654586792, -1.4906392097473145, -0.674454391002655, 0.28473061323165894, 0.4822147786617279, -0.1630537211894989, 0.9504280090332031, 0.6330104470252991, -0.5137242078781128, 0.5479958057403564, 0.020335854962468147, 0.009967767633497715, -1.1614131927490234, 0.5462883710861206, -0.7880293726921082, 1.3739773035049438, -0.2530902624130249, 0.7089292407035828, -0.47866523265838623, -1.7329617738723755, 0.09616553783416748, -1.080626130104065, -0.12727396190166473, -0.473458468914032, 1.998468279838562, -0.2955578863620758, -0.12629269063472748, -0.21651290357112885, 1.6240150928497314, 0.1320359706878662, -0.012143069878220558], [1.658251404762268, 1.3514889478683472, -0.23758548498153687, -1.6142442226409912, 0.6920614242553711, -0.03437412530183792, -0.8116710186004639, 0.4681832492351532, 0.23805251717567444, -1.14543616771698, 0.7579072713851929, 0.3851333260536194, 1.4128944873809814, -0.15053711831569672, 1.481666922569275, 0.03928324207663536, -0.09694691002368927, 0.21055454015731812, 0.663053572177887, -0.8597024083137512, 1.3651424646377563, 0.41317662596702576, 1.0553984642028809, 0.30433326959609985, 0.5872637033462524, 0.36496520042419434, -1.272582769393921, -1.4649875164031982, -0.3037051260471344, 2.099390745162964, -1.024524211883545, 0.5374708771705627, 1.1903682947158813, -0.3412351906299591, 0.6761102676391602, 0.24114982783794403, 2.4482691287994385], [1.1382439136505127, -1.2881072759628296, -0.12346839904785156, -1.2430692911148071, -0.21699480712413788, -0.7931428551673889, -0.9413170218467712, 1.5205082893371582, 0.4639812707901001, 0.5550133585929871, 1.1970884799957275, 1.4076253175735474, 0.38769128918647766, -1.8315939903259277, 1.1457089185714722, 0.49090272188186646, -1.9207549095153809, -0.3983045816421509, -1.0971956253051758, -1.2211004495620728, 0.8784478306770325, -0.7507504820823669, -0.24434620141983032, -0.797267496585846, 1.0816618204116821, -1.5483310222625732, -0.23804832994937897, 0.7953725457191467, -0.1595650017261505, -0.9532778859138489, -1.7055233716964722, 1.0778132677078247, 0.5113649964332581, 0.26265496015548706, -0.12322694808244705, 0.7090061902999878, 0.4844743311405182], [-1.6023626327514648, -0.15406379103660583, 0.9042766094207764, 1.321474313735962, -2.4855082035064697, 0.593003511428833, 1.5207144021987915, 0.3891819715499878, 1.4360448122024536, -0.3310607671737671, 0.13612200319766998, -0.3422800302505493, -0.5126874446868896, 0.6521347165107727, -0.207582488656044, 0.9110899567604065, 0.49944281578063965, -1.1538366079330444, 0.8666517734527588, -2.4529216289520264, -0.47843191027641296, -0.761239767074585, -1.1987131834030151, 0.11800036579370499, 1.2546000480651855, 0.6978268623352051, 0.2236749529838562, -1.3418769836425781, 0.44884997606277466, -0.6310449242591858, -0.627059817314148, 0.24412421882152557, -0.8332173824310303, -0.8175269365310669, 0.27400654554367065, -0.6721097826957703, 0.7799350023269653], [-1.39374577999115, 0.7120146155357361, 0.6049616932868958, 0.11393553018569946, -1.044179081916809, 0.8071915507316589, -0.7431193590164185, 1.0861340761184692, 0.5000050067901611, -1.0356597900390625, 0.39218682050704956, 0.6898097395896912, 0.6598575115203857, 0.07738855481147766, 0.14126180112361908, 1.9041229486465454, 0.4076966345310211, 0.831974983215332, 0.2708197236061096, -1.4095731973648071, 0.6698771119117737, -0.1244528517127037, 0.13897311687469482, -0.13970491290092468, -0.7595763206481934, -0.5479749441146851, -0.7363196611404419, 0.49793359637260437, -0.5911637544631958, -0.3977425694465637, -1.1598536968231201, 1.009255051612854, -0.31409990787506104, -0.032340988516807556, 0.40252944827079773, -0.8871814012527466, 0.00895849708467722], [0.6823328733444214, 2.0640182495117188, 0.1274835169315338, -0.24001550674438477, 0.30037280917167664, -0.32623109221458435, -1.471272349357605, -0.2874336242675781, -0.43077540397644043, 0.2115551084280014, -2.962822914123535, 0.051389098167419434, -0.16773027181625366, 0.12769223749637604, -0.48131000995635986, -0.04161091521382332, -1.7200032472610474, -0.7591290473937988, -1.0925781726837158, 1.0435446500778198, 0.42391878366470337, -0.7029416561126709, -1.9111186265945435, -0.04708694666624069, 0.3846856653690338, -1.225770115852356, -0.05430302023887634, -0.3594968616962433, 0.38306719064712524, 0.040614914149045944, -0.5869288444519043, -0.36226457357406616, 1.3502027988433838, 0.7236049175262451, 1.2389312982559204, 0.20571693778038025, 0.2830997407436371], [1.4340143203735352, -0.46912047266960144, 0.11207376420497894, 0.23327688872814178, -1.345410704612732, 2.077136278152466, -1.7504762411117554, -1.5116926431655884, -0.9712616205215454, 0.1412503719329834, 0.7939398288726807, 1.656175136566162, 0.6266283392906189, -0.7815420031547546, 0.448634535074234, -1.6738845109939575, 1.4807631969451904, -0.18620838224887848, -2.474432945251465, -0.15019400417804718, 0.3649060130119324, -1.3432399034500122, -0.6810644268989563, -0.48705801367759705, -0.1129632219672203, -1.400496482849121, 0.5282371044158936, -0.4505748748779297, -2.4325101375579834, -1.2717516422271729, 1.1666024923324585, 1.5931049585342407, 0.5545772314071655, -0.07320576906204224, -0.6352289319038391, -1.6600384712219238, -1.5916630029678345], [1.3807960748672485, -2.554941177368164, 0.10034830123186111, 1.7389963865280151, 0.4930679202079773, -1.4606010913848877, -0.3583725690841675, 0.5617691278457642, -1.0558785200119019, -0.36367812752723694, 0.9007428288459778, -1.0907111167907715, -0.08036642521619797, 0.4012337327003479, 0.11339008063077927, 0.06693365424871445, 0.7739614248275757, 1.062143325805664, 1.351588249206543, 1.4222829341888428, 0.6195971965789795, -0.5007001757621765, -0.7996721863746643, 0.9505655169487, -0.5395637154579163, -1.3793941736221313, -0.03270253539085388, -0.7613909244537354, -0.22964878380298615, -1.7212369441986084, -0.629080057144165, 1.0884937047958374, 0.08380778133869171, -1.2148098945617676, -0.7494128346443176, 0.657146155834198, -0.0672488808631897], [0.2360672801733017, -1.6583662033081055, -0.0946417823433876, -0.44281938672065735, -0.36208751797676086, 1.8032783269882202, -0.6210978627204895, -0.7923898696899414, -0.2766619920730591, -1.5539727210998535, -0.8935303688049316, -1.4731560945510864, -0.24218299984931946, -0.7271713614463806, -2.022196054458618, -0.743277907371521, -1.2911009788513184, -2.3066892623901367, -0.4399913251399994, 1.3776289224624634, -3.1074860095977783, -0.24723955988883972, 1.4947094917297363, 1.092132806777954, 1.5757731199264526, 2.121615171432495, 0.613091230392456, 1.8505864143371582, 1.642641544342041, -0.1348176896572113, -1.2483021020889282, 2.6666338443756104, 1.6473783254623413, 0.3354478180408478, -0.48592498898506165, -1.2219505310058594, 0.5901097655296326], [0.02994864620268345, -0.47972896695137024, -0.10930299013853073, 0.8156010508537292, 0.40882977843284607, 0.6804147362709045, -1.0313217639923096, 1.977463722229004, 0.8682365417480469, 0.476766973733902, 0.6761230230331421, -0.4697578251361847, 0.6918941736221313, 0.43648919463157654, 0.37197592854499817, 1.9212340116500854, 1.131777048110962, -0.2481715977191925, -0.918412446975708, 0.7967122197151184, -0.3880603015422821, 0.1278446465730667, -0.000260566477663815, -1.3741542100906372, 0.880432665348053, -0.06159878522157669, -0.5102786421775818, 0.41088852286338806, -0.08535782247781754, -0.5184667110443115, -2.5404212474823, 0.2911219894886017, 1.5260764360427856, 0.3171869218349457, 0.33343324065208435, 0.1541818231344223, -0.18920372426509857]], "dtype": "float32"}], "e": {"__nd__": [[0.9886084198951721, 5.273735523223877, -5.393312931060791, -5.850381374359131, 3.113734722137451, 5.117176055908203, -2.3321831226348877, -2.2694785594940186, -10.481300354003906, -5.878067493438721, -4.730775833129883, -4.347259521484375, -0.972137987613678, 11.787773132324219, -10.943242073059082, 0.43961524963378906, -1.6412630081176758, 0.35768720507621765, 2.628471612930298, 12.517684936523438, -12.747121810913086, 1.119719386100769, 5.915123462677002, 6.553383827209473, -2.6059226989746094, -0.5237019658088684, 1.7540568113327026, 12.124373435974121, 0.748606264591217, 4.3449296951293945, 10.117871284484863, 5.0323286056518555, -0.7604349851608276, 0.18184968829154968, 4.1921467781066895, -4.896668434143066, 9.330302238464355], [-14.434285163879395, 3.837967872619629, 8.257418632507324, -8.546599388122559, -19.621641159057617, 2.754678726196289, 2.3561947345733643, -2.9180846214294434, 10.348015785217285, -17.363489151000977, 6.114564895629883, 5.001677989959717, -9.657466888427734, -11.239466667175293, 4.891280651092529, 1.616578221321106, 3.4851865768432617, 0.5487713813781738, 8.259598731994629, -0.5678902268409729, -18.984695434570312, 2.4403414726257324, 3.3700156211853027, -9.509117126464844, -5.501776695251465, 14.548118591308594, 10.648233413696289, -8.136838912963867, 2.309873104095459, 8.14441967010498, 3.498352527618408, -2.1132121086120605, 1.3737406730651855, -8.454933166503906, -1.5223277807235718, -11.695518493652344, 4.893998146057129], [-14.77563762664795, 2.621915817260742, -7.761826515197754, 10.608765602111816, -5.234199523925781, 15.382728576660156, -1.4652228355407715, 4.378421783447266, 15.636022567749023, -14.76504135131836, -10.552459716796875, -1.0246529579162598, -9.041677474975586, 20.578033447265625, -1.9723408222198486, 8.703615188598633, 4.504721164703369, -3.3101863861083984, 10.41242504119873, 11.133432388305664, -6.1898956298828125, -12.072123527526855, -4.296327114105225, -0.14349544048309326, -1.5487884283065796, 12.4323091506958, -2.6613450050354004, -16.41159439086914, 2.721254348754883, 12.524545669555664, -7.549596786499023, -6.416144847869873, 4.552248477935791, 5.734158515930176, -10.142902374267578, 4.292174816131592, -14.07496166229248], [-15.086039543151855, 14.776077270507812, 14.248950004577637, 5.154900550842285, -4.751420021057129, -1.5106191635131836, 7.834533214569092, 12.430148124694824, -6.83582878112793, -9.079277992248535, -2.7216780185699463, 3.5377326011657715, 0.3021104335784912, 0.6270292401313782, -3.5903308391571045, 8.589822769165039, 10.572961807250977, -13.958102226257324, -1.8483271598815918, -13.685914039611816, 14.608535766601562, 0.14522811770439148, -11.000201225280762, -8.982240676879883, -6.863447189331055, -1.684062123298645, 8.572561264038086, -9.41874885559082, 1.9815831184387207, -11.14448356628418, 4.915740013122559, -5.415957927703857, 0.8137509822845459, -1.3757317066192627, -6.022639751434326, -1.2128279209136963, 18.15229034423828], [-8.795917510986328, -1.4532946348190308, -8.94227123260498, -0.9377681016921997, -11.20583438873291, 0.809385359287262, -0.8963103294372559, 10.969325065612793, -3.9652786254882812, -5.022836685180664, 0.9775987863540649, -0.07416050136089325, -4.873012065887451, 11.485820770263672, 8.024399757385254, -1.2294225692749023, 0.6181055307388306, -8.069626808166504, -4.232731342315674, 19.782739639282227, -7.1663055419921875, 5.48476505279541, 2.0641298294067383, -0.9314886927604675, 12.830684661865234, -0.5782046914100647, -8.044480323791504, 5.246629238128662, 10.580907821655273, 4.761150360107422, 6.591682434082031, -7.920776844024658, -1.0126683712005615, -7.261332988739014, -1.5321754217147827, 3.8716037273406982, 18.374210357666016]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(matmul, json.loads(_TESTS), 'cuda', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")